# Demand Planning Tool - Production Forecast v5

This notebook creates a comprehensive demand planning analysis with:
- Historical sales analysis by Product Name/SKU and channel
- **Improved statistical forecasts** using 2-year product-type seasonality + weighted trend
- Seasonal indices calculated at **product type level** (pooled across all SKUs in a type)
- Recency-weighted trend per SKU (recent months weighted more heavily)
- Forecast pivot tabs show last 2 years of actuals alongside forecast for direct comparison
- Export to Google Sheets

## Forecasting Methodology (v5)
- **Seasonality**: Pooled across all SKUs within a product type, using the last 24 months of TOTAL channel data. Seasonal indices are stable because they draw on the full volume of a product type rather than a single item.
- **Trend**: Linear trend via weighted least-squares on each SKU's last 12 months (deseasonalized). Recent months weighted up to 12× heavier.
- **Base Level**: Exponentially weighted moving average of last 12 months per SKU.
- **Manual Growth Override**: Optional annual growth rate applied on top of data-driven trend.

## Setup Instructions
1. Upload your CSV file when prompted
2. Run all cells in order
3. Authenticate with Google when prompted
4. The output will be saved to your Google Drive

In [29]:
# Install required packages
!pip install gspread pandas numpy openpyxl scipy -q

In [30]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
from google.colab import auth
import gspread
from oauth2client.client import GoogleCredentials

In [31]:
# File upload is handled in the next cell.
# Set the filename there, or uncomment the upload block to upload interactively.


In [32]:
# ============================================================
# STEP 2: LOAD SALES DATA
# ============================================================
# Upload your sales CSV file or specify a pre-uploaded filename
# ============================================================

# Option 1: Upload a new sales CSV file (uncomment to use)
uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]

# Option 2: Use a pre-uploaded file (default)
# filename = '_SELECT_mts_created_date_EXTRACT_YEAR_FROM_mts_created_date_100__202603031704.csv'  # Change this to your uploaded filename

print(f"Loading data from: {filename}")

try:
    df = pd.read_csv(filename)

    print(f"✅ Data loaded successfully")
    print(f"   Total rows: {len(df):,}")
    print(f"   Date range: {df['created_date'].min()} to {df['created_date'].max()}")
    print(f"   Channels: {df['orders__source'].unique()}")
    print(f"   Unique SKUs: {df['products__variants__sku'].nunique()}")
    print(f"\nFirst few rows:")
    display(df.head())

except FileNotFoundError:
    print(f"⚠️  File not found: {filename}")
    print("   Please upload the file first, or uncomment the upload section above.")
    raise
except Exception as e:
    print(f"⚠️  Error loading file: {e}")
    raise

Saving anonymized_data.csv to anonymized_data (1).csv
Loading data from: anonymized_data (1).csv
✅ Data loaded successfully
   Total rows: 89,790
   Date range: 2022-01-01 to 2026-02-22
   Channels: ['Direct-to-Consumer' 'Wholesale' 'Kristina Holey']
   Unique SKUs: 277

First few rows:


,created_date,month_num,month_year,products__master__id,products__variants__sku,products__variants__title,products__root_product__title,products__product_type,products__vendor,orders__source,customers__customer_category,quantity,price,total_gross_sales,total_net_sales,total_sales
0,2022-01-01,202201,22-Jan,ID-C6670A7F,SKU-0EB1695C,Product Variant 0001,Product 0001,TYPE_001,VENDOR_001,Direct-to-Consumer,Direct-to-Consumer,7,90.0,630.0,630.0,630.0
1,2022-01-01,202201,22-Jan,ID-63C1C156,SKU-8E520E31,Product Variant 0002,Product 0002,TYPE_002,VENDOR_001,Direct-to-Consumer,Direct-to-Consumer,7,65.0,455.0,447.2,447.2
2,2022-01-01,202201,22-Jan,ID-955F8310,SKU-2EFD6C7A,Product Variant 0003,Product 0003,TYPE_001,VENDOR_001,Direct-to-Consumer,Direct-to-Consumer,4,110.0,440.0,440.0,440.0
3,2022-01-01,202201,22-Jan,ID-9C42608B,SKU-F9E36644,Product Variant 0004,Product 0004,TYPE_002,VENDOR_001,Direct-to-Consumer,Direct-to-Consumer,4,95.0,380.0,380.0,380.0
4,2022-01-01,202201,22-Jan,ID-78BFD7C4,SKU-0475C368,Product Variant 0005,Product 0005,TYPE_001,VENDOR_001,Direct-to-Consumer,Direct-to-Consumer,3,90.0,270.0,270.0,270.0


In [33]:
# Prepare data for analysis
df['created_date'] = pd.to_datetime(df['created_date'])
df['year_month'] = df['created_date'].dt.to_period('M')
df['year'] = df['created_date'].dt.year
df['month'] = df['created_date'].dt.month

# Aggregate to monthly level by SKU and channel
# Include total_net_sales if available for Pareto analysis
agg_dict = {'quantity': 'sum'}
if 'total_net_sales' in df.columns:
    agg_dict['total_net_sales'] = 'sum'
    print("   Including total_net_sales in aggregation")

monthly_data = df.groupby(['year_month', 'products__variants__sku', 'orders__source']).agg(agg_dict).reset_index()
monthly_data['year_month_str'] = monthly_data['year_month'].astype(str)

# Get SKU details
# Use first NON-NULL value for product_type — plain 'first' picks up NaN if
# the first raw row has a blank type, causing seasonal index lookups to
# silently fall back to flat 1.0.
# Also normalise to uppercase so it matches the keys in pt_seasonal_indices.
sku_details = df.groupby('products__variants__sku').agg(
    products__variants__title    =('products__variants__title',    'first'),
    products__root_product__title=('products__root_product__title','first'),
    products__product_type       =('products__product_type',
                                    lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan),
).reset_index()
# Normalise product_type case to uppercase — the seasonal index dict is keyed
# from monthly_data which uses the raw values; normalising both ends ensures
# sku_details.get(prod_type) never misses due to a case difference.
sku_details['products__product_type'] = sku_details['products__product_type'].str.upper()

# Standardize product_name: prefer root product title, fall back to variant title
sku_details['product_name'] = sku_details['products__root_product__title'].fillna(
    sku_details['products__variants__title']
)

print("Data aggregated to monthly level")
print(f"Monthly records: {len(monthly_data):,}")
print(f"\nSample SKU → Product Name mapping:")
print(sku_details[['products__variants__sku', 'product_name', 'products__product_type']].head(10).to_string(index=False))

   Including total_net_sales in aggregation
Data aggregated to monthly level
Monthly records: 7,659

Sample SKU → Product Name mapping:
products__variants__sku product_name products__product_type
           SKU-004F49D5 Product 0160               TYPE_009
           SKU-005A4293 Product 0039               TYPE_005
           SKU-00BCD8BD Product 0141               TYPE_012
           SKU-0307D0F9 Product 0038               TYPE_008
           SKU-03681C19 Product 0138               TYPE_020
           SKU-03BF1380 Product 0003               TYPE_013
           SKU-03CB6550 Product 0091               TYPE_005
           SKU-0475C368 Product 0005               TYPE_001
           SKU-05115970 Product 0136               TYPE_012
           SKU-0580297A Product 0178               TYPE_013


In [34]:
# ============================================================================
# TUNE YOUR FORECAST SETTINGS HERE
# ============================================================================

# ── Actuals cutoff ───────────────────────────────────────────────────────────
# Set to the last month of actuals you have loaded (YYYY-MM).
# The model treats everything up to and including this month as history
# and forecasts from the month after it onward.
#   e.g. '2026-01'  → forecast Feb–Dec 2026  (Jan is latest actual)
#        '2026-02'  → forecast Mar–Dec 2026  (Feb actuals now loaded)
HISTORY_CUTOFF = '2026-02'

# ── Base level method ────────────────────────────────────────────────────────
# Controls how the deseasonalized base level is computed from the history window.
#
#   'mean'    — simple average across the window (recommended; most stable)
#   'median'  — median; more robust if a single month is a big outlier
#   'ewm'     — exponentially weighted mean (favours recent months);
#               only use this if you deliberately want recency bias
BASE_LEVEL_METHOD = 'mean'

# How many months of history to include in the base level calculation (max 24).
# 12 = one full seasonal cycle (recommended default).
# 24 = two years; smooths over an unusual year but may lag a real trend shift.
BASE_LEVEL_WINDOW = 24

# Alpha for EWM only — ignored when BASE_LEVEL_METHOD is 'mean' or 'median'.
# Lower = flatter weights (less recency bias). Range 0.01–0.3.
BASE_LEVEL_ALPHA  = 0.05

# ── Forecast horizon ─────────────────────────────────────────────────────────
# 'rest_of_year'  → forecast through Dec of the cutoff year
# '18_months'     → forecast 18 months starting from the month after cutoff
FORECAST_HORIZON = 'rest_of_year'   # ← flip to '18_months' for extended horizon

# ── Per-channel growth rate overrides ────────────────────────────────────────
# Manual annual growth added on top of the data-driven statistical trend.
# Leave at 0.0 to rely purely on the model's trend for that channel.
#   0.10  = +10% annual growth on top of trend
#   0.0   = pure statistical forecast
#  -0.10  = force 10% annual decline on top of trend
CHANNEL_GROWTH_RATES = {
    'Direct-to-Consumer': 0.0,   # ← tune DTC here
    'Wholesale':          0.0,   # ← tune Wholesale here
    'TOTAL':              0.0,   # ← tune Total here
}

# ── Derived values — do not edit below this line ─────────────────────────────
_cutoff           = pd.Period(HISTORY_CUTOFF, freq='M')
_forecast_start   = _cutoff + 1

if FORECAST_HORIZON == '18_months':
    _forecast_end = _cutoff + 18
else:  # rest_of_year
    _forecast_end = pd.Period(f"{_cutoff.year}-12", freq='M')

forecast_months     = pd.period_range(_forecast_start, _forecast_end, freq='M')
forecast_cal_months = [p.month for p in forecast_months]

# String versions used by downstream cells — single source of truth
FORECAST_START        = str(forecast_months[0])   # e.g. '2026-02'
ACTUALS_HISTORY_START = str(_cutoff - 23)          # 24-month actuals window start, e.g. '2024-02'

print(f"History cutoff        : {HISTORY_CUTOFF}")
print(f"Forecast range        : {FORECAST_START} → {forecast_months[-1]}  ({len(forecast_months)} months)")
print(f"Actuals window start  : {ACTUALS_HISTORY_START}  (24 months before cutoff)")
print(f"Forecast horizon      : {FORECAST_HORIZON}")
print()
print("Per-Channel Growth Rate Overrides:")
for ch, rate in CHANNEL_GROWTH_RATES.items():
    print(f"  {ch:<25} {rate*100:+.1f}% annually")


History cutoff        : 2026-02
Forecast range        : 2026-03 → 2026-12  (10 months)
Actuals window start  : 2024-03  (24 months before cutoff)
Forecast horizon      : rest_of_year

Per-Channel Growth Rate Overrides:
  Direct-to-Consumer        +0.0% annually
  Wholesale                 +0.0% annually
  TOTAL                     +0.0% annually


In [35]:
# ============================================================
# STEP 1: BUILD PRODUCT-TYPE SEASONAL INDICES
# ============================================================
# Seasonality is pooled across ALL SKUs within each product type,
# using the last 24 months of TOTAL channel data.
# Zero-quantity months excluded so new SKU pre-launch zeros don't
# dilute the seasonal signal for their product type.
# ============================================================

MONTH_NAMES = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

def build_product_type_seasonal_indices(monthly_data, sku_details, cutoff='2026-01'):
    """
    Calculate seasonal indices per product type per calendar month.
    Uses TOTAL channel (all channels combined) for the last 24 months up to cutoff.
    Returns a dict: {product_type: {month_number: index}}
    """
    total_monthly = monthly_data.groupby(['year_month_str', 'products__variants__sku'])['quantity'].sum().reset_index()

    total_monthly = total_monthly.merge(
        sku_details[['products__variants__sku', 'products__product_type']],
        on='products__variants__sku', how='left'
    )

    cutoff_period = pd.Period(cutoff, freq='M')
    start_period = cutoff_period - 23
    total_monthly = total_monthly[
        (total_monthly['year_month_str'] >= str(start_period)) &
        (total_monthly['year_month_str'] <= cutoff)
    ].copy()

    # Drop zero-quantity rows — pre-launch months for new SKUs would otherwise
    # dilute the product-type seasonal signal
    total_monthly = total_monthly[total_monthly['quantity'] > 0]

    total_monthly['cal_month'] = total_monthly['year_month_str'].str[5:7].astype(int)

    # Normalise product_type to uppercase so keys always match sku_details
    total_monthly['products__product_type'] = total_monthly['products__product_type'].str.upper()
    pt_monthly = total_monthly.groupby(['products__product_type', 'cal_month'])['quantity'].sum().reset_index()

    seasonal_indices = {}
    product_types = pt_monthly['products__product_type'].unique()

    for pt in product_types:
        pt_data = pt_monthly[pt_monthly['products__product_type'] == pt].copy()

        month_qty = pt_data.set_index('cal_month')['quantity'].astype(float).to_dict()
        month_qty = {int(k): v for k, v in month_qty.items()}

        if len(month_qty) > 0:
            avg_qty = np.mean(list(month_qty.values()))
        else:
            avg_qty = 1.0

        for m in range(1, 13):
            if m not in month_qty:
                month_qty[m] = avg_qty

        grand_avg = np.mean([month_qty[m] for m in range(1, 13)])
        if grand_avg > 0:
            indices = {m: month_qty[m] / grand_avg for m in range(1, 13)}
        else:
            indices = {m: 1.0 for m in range(1, 13)}

        seasonal_indices[pt] = indices

    return seasonal_indices


# Build product-type seasonal indices
pt_seasonal_indices = build_product_type_seasonal_indices(monthly_data, sku_details, cutoff=HISTORY_CUTOFF)

print(f"\u2705 Seasonal indices built for {len(pt_seasonal_indices)} product types")
print("\nProduct types found:")
for pt in sorted(pt_seasonal_indices.keys()):
    print(f"  \u2022 {pt}")

# Diagnostic: flag SKUs with missing or unmatched product_type
_pt_known    = set(pt_seasonal_indices.keys())
_pt_problems = sku_details[
    sku_details['products__product_type'].isna() |
    ~sku_details['products__product_type'].isin(_pt_known)
][['products__variants__sku', 'products__product_type']]
if len(_pt_problems) > 0:
    print(f"\n\u26a0\ufe0f  {len(_pt_problems)} SKUs have missing/unmatched product_type — will get flat 1.0 indices:")
    print(_pt_problems.to_string(index=False))
else:
    print("\u2705 All SKU product types matched to seasonal indices")

# ── Diagnostic: flag SKUs with missing or unmatched product_type ─────────────
_pt_known    = set(pt_seasonal_indices.keys())
_pt_problems = sku_details[
    sku_details['products__product_type'].isna() |
    ~sku_details['products__product_type'].isin(_pt_known)
][['products__variants__sku', 'products__product_type']]
if len(_pt_problems) > 0:
    print(f"\n\u26a0\ufe0f  {len(_pt_problems)} SKUs have a missing/unmatched product_type — will get flat 1.0 indices:")
    print(_pt_problems.to_string(index=False))
else:
    print("\u2705 All SKU product types matched to seasonal indices")


✅ Seasonal indices built for 26 product types

Product types found:
  • TYPE_001
  • TYPE_002
  • TYPE_003
  • TYPE_004
  • TYPE_005
  • TYPE_006
  • TYPE_007
  • TYPE_008
  • TYPE_009
  • TYPE_010
  • TYPE_011
  • TYPE_012
  • TYPE_013
  • TYPE_014
  • TYPE_015
  • TYPE_016
  • TYPE_017
  • TYPE_018
  • TYPE_019
  • TYPE_020
  • TYPE_021
  • TYPE_022
  • TYPE_023
  • TYPE_024
  • TYPE_025
  • TYPE_026
✅ All SKU product types matched to seasonal indices
✅ All SKU product types matched to seasonal indices


In [36]:
# ============================================================
# STEP 2: DISPLAY SEASONALITY TABLE (Product Type × Month)
# ============================================================
# This shows the seasonal index for each product type by month.
# Index > 1.0 = that month is stronger than average
# Index < 1.0 = that month is weaker than average
# ============================================================

seasonality_rows = []
for pt, indices in sorted(pt_seasonal_indices.items()):
    row = {'Product Type': pt}
    for m in range(1, 13):
        row[MONTH_NAMES[m]] = round(indices[m], 3)
    seasonality_rows.append(row)

seasonality_display_df = pd.DataFrame(seasonality_rows)

print("📊 SEASONAL INDICES BY PRODUCT TYPE AND MONTH")
print("   (Based on last 24 months of TOTAL channel data)")
print("   Index > 1.0 = stronger than annual average | Index < 1.0 = weaker than annual average")
print()
print(seasonality_display_df.to_string(index=False))
print()
print("Note: Indices within each product type sum to 12.0 (average = 1.0)")

📊 SEASONAL INDICES BY PRODUCT TYPE AND MONTH
   (Based on last 24 months of TOTAL channel data)
   Index > 1.0 = stronger than annual average | Index < 1.0 = weaker than annual average

Product Type   Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct   Nov   Dec
    TYPE_001 0.897 0.855 1.019 0.803 0.777 0.763 0.996 1.041 1.370 0.708 1.961 0.810
    TYPE_002 0.955 0.910 1.009 0.849 0.854 0.781 0.949 1.030 1.267 0.721 1.851 0.823
    TYPE_003 0.978 0.829 1.058 0.805 0.898 0.897 1.048 0.939 1.239 0.822 1.592 0.894
    TYPE_004 0.807 0.866 0.937 0.774 0.795 0.978 1.037 1.083 1.227 0.889 1.855 0.752
    TYPE_005 1.000 1.000 1.371 0.971 1.020 1.156 0.947 1.174 1.027 0.934 0.400 1.000
    TYPE_006 1.000 0.009 0.508 0.491 0.861 0.250 5.830 0.009 1.000 1.000 1.000 0.043
    TYPE_007 0.748 0.708 0.894 1.053 1.295 1.015 1.121 0.944 1.055 0.606 1.788 0.771
    TYPE_008 0.898 0.551 0.852 0.753 1.434 0.749 1.122 1.093 1.365 0.810 1.495 0.877
    TYPE_009 0.612 0.703 0.943 1.100 0.827 1.274 

In [37]:
# ============================================================
# STEP 3: FORECASTING ENGINE
# ============================================================

# ── Trend stability controls (tune in config cell if needed) ─────────────
# MIN_MONTHS_FOR_TREND: minimum non-zero months before any trend is applied.
# Below this threshold the forecast is flat-seasonal (base × seasonal index).
MIN_MONTHS_FOR_TREND         = 6
TREND_SLOPE_CAP_PCT          = 0.05   # Top Performers: max 5%/month trend
TREND_SLOPE_CAP_PCT_BOTTOM20 = 0.02   # Bottom 20%: tighter 2%/month cap


def exponential_weighted_mean(arr, alpha=0.08):
    """Exponentially weighted mean — most recent observation has highest weight."""
    n = len(arr)
    weights = np.array([(1 - alpha) ** (n - 1 - i) for i in range(n)])
    return np.dot(weights, arr) / weights.sum()


def compute_base_level(deseason_values, method='mean'):
    """
    Compute the deseasonalized base level.
    Zero-sale months are excluded so new SKUs are not penalised by
    pre-launch zeros. Returns 0.0 if every value is zero.

    Methods: 'mean' (default), 'median', 'ewm'
    """
    nonzero_values = deseason_values[deseason_values > 0]
    if len(nonzero_values) == 0:
        return 0.0
    if method == 'median':
        return float(np.median(nonzero_values))
    elif method == 'ewm':
        return exponential_weighted_mean(nonzero_values, alpha=BASE_LEVEL_ALPHA)
    else:  # 'mean' — default
        return float(np.mean(nonzero_values))


def calculate_weighted_trend(deseasonalized_values):
    """
    Estimate trend (units/month) from year-over-year annual averages.

    Instead of fitting WLS across the raw monthly sequence — which produces
    a falsely negative slope when the window ends on a weak month (e.g. Feb)
    even for a flat or growing SKU — we bucket the history into 12-month
    blocks, compute the mean deseasonalized volume per block, then fit a
    weighted line through those annual averages.

    This makes the trend immune to seasonal position at the cutoff date.
    Returns 0.0 if fewer than 2 yearly buckets of non-zero data exist.
    """
    first_nonzero = next((i for i, v in enumerate(deseasonalized_values) if v > 0), None)
    if first_nonzero is None:
        return 0.0
    deseasonalized_values = deseasonalized_values[first_nonzero:]

    n = len(deseasonalized_values)
    if n < 4:
        return 0.0

    # Split into 12-month yearly buckets (oldest first)
    n_years = max(1, int(np.ceil(n / 12)))
    if n_years < 2:
        return 0.0   # need at least 2 years to estimate a cross-year trend

    bucket_means = []
    for yr in range(n_years):
        chunk = deseasonalized_values[yr * 12: (yr + 1) * 12]
        nonzero = chunk[chunk > 0]
        if len(nonzero) > 0:
            bucket_means.append((float(yr), float(np.mean(nonzero))))

    if len(bucket_means) < 2:
        return 0.0

    xs   = np.array([b[0] for b in bucket_means])
    ys   = np.array([b[1] for b in bucket_means])
    ws   = xs + 1.0   # weight recent years more

    wx   = (ws * xs).sum();  wy  = (ws * ys).sum()
    wxx  = (ws * xs * xs).sum();  wxy = (ws * xs * ys).sum()
    wsum = ws.sum()

    denom = wsum * wxx - wx * wx
    if abs(denom) < 1e-10:
        return 0.0

    # Slope is units per yearly bucket → divide by 12 to get units/month
    return (wsum * wxy - wx * wy) / denom / 12.0


def calculate_forecast(
    historical_values,
    calendar_months,
    product_type,
    pt_seasonal_indices,
    forecast_calendar_months,
    growth_rate,
    pareto_category='Top Performers',
):
    """
    Forecast using product-type seasonal indices.

    Stability rules for low-volume / new SKUs:
      1. Trend applied only when >= MIN_MONTHS_FOR_TREND active months exist.
      2. Trend slope capped at ±TREND_SLOPE_CAP_PCT × base_level per month.
         Bottom-20% SKUs use the tighter TREND_SLOPE_CAP_PCT_BOTTOM20 cap.
      3. Projected base floored at 20% of base_level to prevent near-zero decay.
    """
    n = len(historical_values)
    seasonal_indices = pt_seasonal_indices.get(product_type, {m: 1.0 for m in range(1, 13)})

    if n < 4:
        nonzero = historical_values[historical_values > 0]
        avg = np.mean(nonzero) if len(nonzero) > 0 else 0
        return [int(round(max(0, avg * seasonal_indices.get(m, 1.0)))) for m in forecast_calendar_months]

    window        = min(n, BASE_LEVEL_WINDOW)
    recent_vals   = historical_values[-window:]
    recent_months = calendar_months[-window:]

    deseason_recent = np.array([
        v / seasonal_indices.get(int(m), 1.0) if seasonal_indices.get(int(m), 1.0) > 0 else v
        for v, m in zip(recent_vals, recent_months)
    ])

    base_level    = compute_base_level(deseason_recent, method=BASE_LEVEL_METHOD)
    active_months = int(np.sum(deseason_recent > 0))

    if active_months < MIN_MONTHS_FOR_TREND:
        trend_slope = 0.0
    else:
        trend_slope = calculate_weighted_trend(deseason_recent)
        if base_level > 0:
            cap_pct   = TREND_SLOPE_CAP_PCT_BOTTOM20 if pareto_category == 'Bottom 20%' else TREND_SLOPE_CAP_PCT
            max_slope = base_level * cap_pct
            trend_slope = float(np.clip(trend_slope, -max_slope, max_slope))

    base_floor     = base_level * 0.20
    monthly_growth = (1 + growth_rate) ** (1 / 12) - 1

    forecasts = []
    for i, cal_month in enumerate(forecast_calendar_months):
        projected_base  = max(base_level + trend_slope * (i + 1), base_floor)
        projected_base *= (1 + monthly_growth) ** (i + 1)
        forecast        = projected_base * seasonal_indices.get(int(cal_month), 1.0)
        forecasts.append(int(round(max(0, forecast))))

    return forecasts


print("\u2705 Forecasting engine loaded (trend-stability + new-SKU fixes)")


✅ Forecasting engine loaded (trend-stability + new-SKU fixes)


In [38]:
# Active SKU List (hardcoded)
ACTIVE_SKUS = [
    'SKU-0EB1695C', 'SKU-8E520E31', 'SKU-2EFD6C7A', 'SKU-F9E36644', 'SKU-0475C368',
    'SKU-0D6CC04A', 'SKU-52426351', 'SKU-477770DD', 'SKU-2C2E1146', 'SKU-340B7492',
    'SKU-7288D24C', 'SKU-9F53CB85', 'SKU-31B18550', 'SKU-82450F9A', 'SKU-12458965',
    'SKU-F52C63A6', 'SKU-C06010BD', 'SKU-30CE4A55', 'SKU-E8CCB5BA', 'SKU-248AA4AB',
    'SKU-61433EE8', 'SKU-CE95D3ED', 'SKU-17E8A790', 'SKU-65C98C86', 'SKU-94071812',
    'SKU-12418161', 'SKU-F43453AE', 'SKU-856919F1', 'SKU-0C90C79E', 'SKU-1B4914B3',
    'SKU-7349C7A4', 'SKU-7EACBAB9', 'SKU-910D4898', 'SKU-CA462E88', 'SKU-34E61ED4',
    'SKU-B53669B2', 'SKU-D78AA18F', 'SKU-FB2D29D0', 'SKU-FB831F26', 'SKU-9D558593',
    'SKU-176738C5', 'SKU-DA4859AE', 'SKU-6FF9499E', 'SKU-26AFF8B1', 'SKU-B1B85544',
    'SKU-F7D382FD', 'SKU-51C4C17E', 'SKU-36825476', 'SKU-AED4F00A', 'SKU-0307D0F9',
    'SKU-05A932C2', 'SKU-005A4293', 'SKU-9D3A83AE', 'SKU-BAB5279F', 'SKU-9AA8EBC1',
    'SKU-D8820FD7', 'SKU-25F5258A', 'SKU-755F05D1', 'SKU-99AADAF0', 'SKU-46F4BEF5',
    'SKU-F623240A', 'SKU-5B871ADC', 'SKU-DD363F7F', 'SKU-4E609532', 'SKU-ED62B166',
    'SKU-23791320', 'SKU-C45CA418', 'SKU-95C2BF50', 'SKU-B8ACD4BB', 'SKU-CB5307D9',
    'SKU-B4C36A20', 'SKU-3FB3363E', 'SKU-170AB515', 'SKU-2BF342A0', 'SKU-ECD2FAFA',
    'SKU-883BFCBB', 'SKU-30B0D862', 'SKU-84C921F4', 'SKU-37BD99A6', 'SKU-95980917',
    'SKU-694B2D87', 'SKU-B12882DB', 'SKU-6401C8F3', 'SKU-DAF40191', 'SKU-E6A8FF00',
    'SKU-5F02AA37', 'SKU-85744951', 'SKU-D8A2CA5A', 'SKU-5DB1D376', 'SKU-EA51728B',
    'SKU-1A00B89C', 'SKU-3BEEA810', 'SKU-D721E23D', 'SKU-37896138', 'SKU-F2583A60',
    'SKU-5EC38CF1', 'SKU-65E51C8A', 'SKU-0DB8736F', 'SKU-A8988E1F', 'SKU-C1680BB3',
    'SKU-7E8FDB84', 'SKU-D3A57DD5', 'SKU-E2BEB294', 'SKU-A0E4F03A', 'SKU-818DC344',
    'SKU-88F18DA5', 'SKU-28B90F36', 'SKU-E7D0F1DA', 'SKU-0A123387', 'SKU-736F2EAC',
    'SKU-D4E0765E', 'SKU-8714BACD', 'SKU-123FFB4F', 'SKU-46075AA6', 'SKU-A0752C4A',
    'SKU-D67B401B', 'SKU-75E69338', 'SKU-E34456B3', 'SKU-CA375611', 'SKU-C45F18D8',
    'SKU-DAB7C4FC', 'SKU-60740245', 'SKU-BA5EC1CE', 'SKU-30C795B8', 'SKU-C43680FA',
    'SKU-C11A3611', 'SKU-EE9D8C7B', 'SKU-9D0A26E3', 'SKU-8E9DDD48', 'SKU-36D9822E',
    'SKU-203B3E3A', 'SKU-226BC0C1', 'SKU-CF833025', 'SKU-34A577C3', 'SKU-460F4F7F',
    'SKU-96ED2A5B', 'SKU-FC51D3D5', 'SKU-CD4AF8DC', 'SKU-A1B47AC8', 'SKU-36C94A3C',
    'SKU-0A9DF7A6', 'SKU-03CB6550', 'SKU-60A2C260', 'SKU-189669BB', 'SKU-38E0D387',
    'SKU-9F5C9CEF', 'SKU-058F47E8', 'SKU-F4FF14B9', 'SKU-DFDA7CC5', 'SKU-2C52E939',
    'SKU-D4FAA20A', 'SKU-38DBEA7A', 'SKU-1E09E5A9', 'SKU-50BB7750', 'SKU-BDE10C3B',
    'SKU-79C1869C', 'SKU-51EAA5B2', 'SKU-2E36B0E8', 'SKU-5BB2DCC4', 'SKU-BAFDEB7B',
    'SKU-14626E65', 'SKU-424AF1D2', 'SKU-A39D2B4A', 'SKU-3C40AE55', 'SKU-19DCF054',
    'SKU-15FA1ED8', 'SKU-F9E3906F', 'SKU-D866E33D', 'SKU-C48F7580', 'SKU-9A208EA3',
    'SKU-FB715387', 'SKU-F4D89558', 'SKU-C118AAC9', 'SKU-96D96431', 'SKU-789A0FB7',
    'SKU-FA74FBC7', 'SKU-A91C4690', 'SKU-D2EA05E9', 'SKU-A6B870D2', 'SKU-5BDFEBFE',
    'SKU-7E449822', 'SKU-6C0B20D2', 'SKU-1716B30A', 'SKU-AC46AA33', 'SKU-E1AA6F1F',
    'SKU-78C0530A', 'SKU-B4030C6B', 'SKU-F4A9A785', 'SKU-F0F50B4D', 'SKU-D7DDAC8B',
    'SKU-05115970', 'SKU-4DFB9C52', 'SKU-03681C19', 'SKU-6CDB7E50', 'SKU-DCB7CF68',
    'SKU-39794ACA', 'SKU-00BCD8BD', 'SKU-9C185612', 'SKU-28D10342', 'SKU-5E0AD0C5',
    'SKU-89DBEF46', 'SKU-5BA041B7', 'SKU-9D248255', 'SKU-6CD94412', 'SKU-98559DFC',
    'SKU-491B6988', 'SKU-058CDED1', 'SKU-79099EA2', 'SKU-D7390F97', 'SKU-13E10651',
    'SKU-D29F7BE4', 'SKU-CAE5C68A', 'SKU-59EB6E6A', 'SKU-4CF51F18', 'SKU-38A96D40',
    'SKU-CAE27696', 'SKU-AEB09AE5', 'SKU-2F918DE3', 'SKU-91665C74', 'SKU-C1B32491',
    'SKU-F923E995', 'SKU-59A12982', 'SKU-8A6A4851', 'SKU-074A6EBD', 'SKU-C35ECA29',
    'SKU-8C921ED1', 'SKU-079D0B6D', 'SKU-004F49D5', 'SKU-24C2F8C0', 'SKU-2D473ADF',
    'SKU-966560F9', 'SKU-50B11E0C', 'SKU-A3F34BF1', 'SKU-2D4AF1D1', 'SKU-A0E1CB1B',
    'SKU-18095882', 'SKU-FBE80D84', 'SKU-2DCD1A80', 'SKU-84BB013B', 'SKU-6AB7B06E'
]

active_skus_from_file = set(ACTIVE_SKUS)
print(f"Active SKUs loaded (hardcoded): {len(active_skus_from_file)}")
print(f"Sample: {list(active_skus_from_file)[:5]}")


Active SKUs loaded (hardcoded): 240
Sample: ['SKU-9AA8EBC1', 'SKU-95C2BF50', 'SKU-D29F7BE4', 'SKU-0C90C79E', 'SKU-EE9D8C7B']


In [39]:
# ============================================================
# PROXY SKU MAPPING (optional)
# ============================================================
# For new SKUs with limited history, specify a proxy SKU whose
# historical volumes will be used to fill the pre-launch gap.
#
# Upload a CSV with two columns:
#   new_sku   — the SKU that needs gap-filling
#   proxy_sku — the established SKU whose history to borrow
#
# The proxy volumes are scaled: proxy × (new_SKU_avg / proxy_avg)
# so the volume level matches the new SKU, not the proxy.
# Real data always wins — proxy only fills months with no real sales.
#
# To skip, just press Cancel when the upload dialog appears.
# ============================================================

print("📂 Upload proxy SKU mapping CSV (optional — cancel to skip)")
print("   Expected columns: new_sku, proxy_sku")
print()

proxy_map = {}   # {new_sku: proxy_sku}

try:
    proxy_files = files.upload()
    if proxy_files:
        import io
        proxy_filename = list(proxy_files.keys())[0]
        proxy_df = pd.read_csv(io.BytesIO(proxy_files[proxy_filename]))
        proxy_df.columns = [c.strip().lower() for c in proxy_df.columns]

        if 'new_sku' in proxy_df.columns and 'proxy_sku' in proxy_df.columns:
            proxy_map = dict(zip(
                proxy_df['new_sku'].astype(str).str.strip(),
                proxy_df['proxy_sku'].astype(str).str.strip()
            ))
            print(f"✅ Proxy mapping loaded: {len(proxy_map)} SKU pairs")
            for new, prx in proxy_map.items():
                print(f"   {new} → {prx}")
        else:
            print(f"⚠️  CSV must have columns 'new_sku' and 'proxy_sku'")
            print(f"   Found: {list(proxy_df.columns)}")
    else:
        print("⏭️  No file uploaded — proxy mapping disabled")
except Exception as e:
    print(f"⏭️  Skipped proxy upload ({e})")

print(f"\nProxy map active for {len(proxy_map)} SKUs")


def build_history_with_proxy(sku, monthly_data_total, proxy_map, history_cutoff):
    """
    Build a complete monthly history array for a SKU, filling pre-launch
    gaps with scaled proxy history where real data is absent.

    Parameters
    ----------
    sku               : str — target SKU
    monthly_data_total: DataFrame with year_month_str + products__variants__sku + quantity
                        (already summed across channels / TOTAL)
    proxy_map         : dict {new_sku: proxy_sku}
    history_cutoff    : str e.g. '2026-02'

    Returns
    -------
    hist_values    : np.array of monthly quantities (chronological)
    hist_cal_months: np.array of calendar month numbers (1-12)
    """
    # Real history for this SKU up to cutoff
    real = monthly_data_total[
        (monthly_data_total['products__variants__sku'] == sku) &
        (monthly_data_total['year_month_str'] <= history_cutoff)
    ].sort_values('year_month_str')

    real_months_set = set(real['year_month_str'].values)

    proxy_sku = proxy_map.get(sku)
    if not proxy_sku:
        # No proxy — return real history as-is
        hist_values     = real['quantity'].values.astype(float)
        hist_cal_months = np.array([int(ym[5:7]) for ym in real['year_month_str'].values])
        return hist_values, hist_cal_months

    # Proxy history up to cutoff
    proxy_real = monthly_data_total[
        (monthly_data_total['products__variants__sku'] == proxy_sku) &
        (monthly_data_total['year_month_str'] <= history_cutoff)
    ].sort_values('year_month_str')

    if len(proxy_real) == 0:
        # Proxy has no data either — fall back to real only
        hist_values     = real['quantity'].values.astype(float)
        hist_cal_months = np.array([int(ym[5:7]) for ym in real['year_month_str'].values])
        return hist_values, hist_cal_months

    # Compute scaling factor: new_SKU_avg / proxy_avg (non-zero months only)
    new_nonzero  = real[real['quantity'] > 0]['quantity'].values
    prx_nonzero  = proxy_real[proxy_real['quantity'] > 0]['quantity'].values

    if len(new_nonzero) > 0 and len(prx_nonzero) > 0:
        scale = float(np.mean(new_nonzero)) / float(np.mean(prx_nonzero))
    else:
        scale = 1.0   # can't compute ratio — use raw proxy

    # Build a month-keyed dict of real values
    real_by_month = {
        row['year_month_str']: row['quantity']
        for _, row in real.iterrows()
    }

    # Build combined history: all months covered by proxy, real data wins
    proxy_by_month = {
        row['year_month_str']: row['quantity'] * scale
        for _, row in proxy_real.iterrows()
    }

    # Union of all months present in either dataset
    all_months_str = sorted(set(real_by_month.keys()) | set(proxy_by_month.keys()))

    combined_values = []
    combined_cal    = []
    for ym in all_months_str:
        if ym in real_by_month:
            # Real data wins
            combined_values.append(float(real_by_month[ym]))
        else:
            # Gap — fill with scaled proxy
            combined_values.append(float(proxy_by_month[ym]))
        combined_cal.append(int(ym[5:7]))

    return np.array(combined_values), np.array(combined_cal)


📂 Upload proxy SKU mapping CSV (optional — cancel to skip)
   Expected columns: new_sku, proxy_sku



Saving proxy - proxy_sku (1).csv to proxy - proxy_sku (1) (1).csv
✅ Proxy mapping loaded: 8 SKU pairs
   FG-10082 → FG-10018
   FG-10104 → FG-10023
   FG-90102 → FG-10040
   FG-90100 → FG-10040
   FG-10099 → FG-10004
   FG-10103 → FG-10044
   FG-10094 → FG-10081
   FG-10083 → FG-10014L

Proxy map active for 8 SKUs


In [40]:
# ============================================================
# PARETO ANALYSIS (80/20 by Revenue)
# ============================================================

print("\nBuilding Pareto Analysis (80/20 by revenue)...")

# Check if total_net_sales column exists
if 'total_net_sales' in monthly_data.columns:
    revenue_col = 'total_net_sales'
    print("   Using 'total_net_sales' column for revenue")
elif 'line_item_price' in monthly_data.columns:
    # Calculate revenue as quantity * price
    monthly_data['calculated_revenue'] = monthly_data['quantity'] * monthly_data['line_item_price']
    revenue_col = 'calculated_revenue'
    print("   Calculated revenue as quantity × line_item_price")
else:
    # Fallback: use quantity as proxy
    monthly_data['calculated_revenue'] = monthly_data['quantity']
    revenue_col = 'calculated_revenue'
    print("   ⚠️  No revenue columns found - using quantity as proxy")

# Filter to active SKUs only
active_monthly = monthly_data[monthly_data['products__variants__sku'].isin(active_skus_from_file)].copy()

# Group by SKU and sum revenue
sku_revenue = active_monthly.groupby('products__variants__sku').agg(
    total_revenue=(revenue_col, 'sum'),
    total_quantity=('quantity', 'sum')
).reset_index()

# Merge with SKU details
sku_revenue = sku_revenue.merge(
    sku_details[['products__variants__sku', 'product_name', 'products__product_type']],
    on='products__variants__sku',
    how='left'
)

# Sort by revenue descending
sku_revenue = sku_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

# Calculate cumulative percentage
total_rev = sku_revenue['total_revenue'].sum()
sku_revenue['revenue_pct'] = (sku_revenue['total_revenue'] / total_rev * 100).round(2)
sku_revenue['cumulative_pct'] = sku_revenue['revenue_pct'].cumsum().round(2)

# Classify: Top Performers = cumulative <= 80%
sku_revenue['Pareto_Category'] = sku_revenue['cumulative_pct'].apply(
    lambda x: 'Top Performers' if x <= 80 else 'Bottom 20%'
)

# Create lookup dict
pareto_lookup = dict(zip(sku_revenue['products__variants__sku'], sku_revenue['Pareto_Category']))

# Stats
top_performers = sku_revenue[sku_revenue['Pareto_Category'] == 'Top Performers']
print(f"✅ Pareto Analysis complete")
print(f"   Total Active SKUs: {len(sku_revenue)}")
print(f"   Top Performers (80% of revenue): {len(top_performers)} SKUs")
print(f"   Bottom 20%: {len(sku_revenue) - len(top_performers)} SKUs")
if revenue_col == 'total_net_sales':
    print(f"   Total Revenue: ${total_rev:,.2f}")
else:
    print(f"   Total Revenue (calculated): ${total_rev:,.2f}")

# Prepare Pareto sheet data
pareto_sheet_data = sku_revenue[[
    'products__variants__sku', 'product_name', 'products__product_type',
    'total_revenue', 'total_quantity', 'revenue_pct', 'cumulative_pct', 'Pareto_Category'
]].copy()
pareto_sheet_data.columns = [
    'SKU', 'Product Name', 'Product Type',
    'Total Revenue', 'Total Quantity', 'Revenue %', 'Cumulative %', 'Pareto Category'
]

# Replace NaN/inf with 0 — Google Sheets API rejects non-JSON-compliant floats
import math
def _safe(v):
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
        return 0
    return v

pareto_sheet_data = pareto_sheet_data.apply(
    lambda col: col.map(_safe) if col.dtype == float else col
)


Building Pareto Analysis (80/20 by revenue)...
   Using 'total_net_sales' column for revenue
✅ Pareto Analysis complete
   Total Active SKUs: 240
   Top Performers (80% of revenue): 19 SKUs
   Bottom 20%: 221 SKUs
   Total Revenue: $32,756,851.88


In [41]:
# ============================================================
# STEP 4: GENERATE FORECASTS
# ============================================================

all_channels = ['Direct-to-Consumer', 'Wholesale', 'TOTAL']

print(f"\nActive SKU filter: {len(active_skus_from_file)} SKUs")

all_data_skus = set(sku_details['products__variants__sku'].unique())
matching = active_skus_from_file & all_data_skus
print(f"Matching SKUs (in both filter and data): {len(matching)}")

if len(matching) == 0:
    print("⚠️  WARNING: NO MATCHING SKUs!")
    print("   Proceeding to forecast ALL SKUs.")
    active_skus_from_file = set()

forecast_results = []
skipped_counts = {ch: 0 for ch in all_channels}
forecasted_skus = set()

_total_channel_data = monthly_data.groupby(
    ['year_month_str', 'products__variants__sku'])['quantity'].sum().reset_index()

_sku_info_lookup = sku_details.set_index('products__variants__sku').to_dict('index')

for channel in all_channels:
    print(f"\nGenerating forecasts for {channel}...")

    if channel == 'TOTAL':
        channel_data = _total_channel_data
    else:
        channel_data = monthly_data[monthly_data['orders__source'] == channel].copy()

    channel_sku_groups = {
        sku: grp for sku, grp in channel_data.groupby('products__variants__sku')
    }

    for sku, sku_data in channel_sku_groups.items():
        sku_info_dict = _sku_info_lookup.get(sku)
        if sku_info_dict is None:
            continue

        class _Row:
            def __getitem__(self, k): return sku_info_dict[k]
            def get(self, k, d=None): return sku_info_dict.get(k, d)
        sku_info = _Row()
        product_type = sku_info_dict['products__product_type']

        if len(active_skus_from_file) > 0 and sku not in active_skus_from_file:
            skipped_counts[channel] += 1
            continue

        # Build history — fills pre-launch gaps with scaled proxy if mapping exists
        hist_values, hist_cal_months = build_history_with_proxy(
            sku, _total_channel_data, proxy_map, HISTORY_CUTOFF
        )
        # For non-TOTAL channels, we still need channel-specific history for the
        # channel forecast; use channel data directly, proxy fills TOTAL only
        if channel != 'TOTAL':
            historical = sku_data[sku_data['year_month_str'] <= HISTORY_CUTOFF].sort_values('year_month_str')
            hist_values     = historical['quantity'].values.astype(float)
            hist_cal_months = np.array([int(ym[5:7]) for ym in historical['year_month_str'].values])

        pareto_cat = pareto_lookup.get(sku, 'Top Performers')

        forecasts = calculate_forecast(
            hist_values,
            hist_cal_months,
            product_type,
            pt_seasonal_indices,
            forecast_cal_months,
            CHANNEL_GROWTH_RATES.get(channel, 0.0),
            pareto_category=pareto_cat,
        )

        for month, forecast_qty in zip(forecast_months, forecasts):
            forecast_results.append({
                'channel': channel,
                'product_name': sku_info['product_name'],
                'sku': sku,
                'product_type': product_type,
                'month': str(month),
                'forecast_qty': forecast_qty
            })

        if channel == 'TOTAL':
            forecasted_skus.add(sku)

forecast_df = pd.DataFrame(forecast_results)
print(f"\n✅ Forecasts generated: {len(forecast_df):,} records")
print(f"   Unique SKUs forecasted (TOTAL channel): {len(forecasted_skus)}")
print(f"   SKUs skipped per channel: { {k: v for k, v in skipped_counts.items()} }")

if len(active_skus_from_file) > 0:
    print(f"\n📋 Filter was applied: {len(active_skus_from_file)} SKUs from file")
    print(f"   {len(forecasted_skus)} SKUs actually forecasted")
    if len(forecasted_skus) != len(matching):
        print(f"   ⚠️  Expected {len(matching)} forecasted SKUs (matching count)")
else:
    print(f"\n⚠️  No filter applied - all {len(forecasted_skus)} SKUs in data were forecasted")



Active SKU filter: 240 SKUs
Matching SKUs (in both filter and data): 240

Generating forecasts for Direct-to-Consumer...

Generating forecasts for Wholesale...

Generating forecasts for TOTAL...

✅ Forecasts generated: 5,580 records
   Unique SKUs forecasted (TOTAL channel): 240
   SKUs skipped per channel: {'Direct-to-Consumer': 30, 'Wholesale': 22, 'TOTAL': 37}

📋 Filter was applied: 240 SKUs from file
   240 SKUs actually forecasted


In [42]:
# ============================================================
# STEP 5: BUILD FORECAST COMPARISON PIVOT
# (2024 actuals | 2025 actuals | 2026 forecast — all months)
# ============================================================

def build_comparison_pivot(channel, monthly_data, forecast_df, sku_details):
    """
    Build a pivot table showing:
      Rows: Product Name | SKU
      Columns: <ACTUALS_HISTORY_START> ... <HISTORY_CUTOFF> (actuals), <FORECAST_START> ... (forecast)
    Adds a row-level label prefix so actuals vs forecast are clear.
    """
    # --- Historical actuals ---
    if channel == 'TOTAL':
        hist_data = monthly_data.groupby(['year_month_str', 'products__variants__sku'])['quantity'].sum().reset_index()
    else:
        hist_data = monthly_data[monthly_data['orders__source'] == channel].copy()
        hist_data = hist_data[['year_month_str', 'products__variants__sku', 'quantity']]

    # Keep actuals from 24-month window up to and including HISTORY_CUTOFF
    hist_data = hist_data[
        (hist_data['year_month_str'] >= ACTUALS_HISTORY_START) &
        (hist_data['year_month_str'] <= HISTORY_CUTOFF)
    ].copy()

    hist_data = hist_data.merge(
        sku_details[['products__variants__sku', 'product_name']],
        on='products__variants__sku', how='left'
    )
    hist_data['type_flag'] = 'Actual'

    # --- Forecast (FORECAST_START onward) ---
    fcst_data = forecast_df[forecast_df['channel'] == channel][['month', 'sku', 'product_name', 'forecast_qty']].copy()
    fcst_data.columns = ['year_month_str', 'products__variants__sku', 'product_name', 'quantity']
    fcst_data['type_flag'] = 'Forecast'

    # Combine
    combined = pd.concat([hist_data, fcst_data], ignore_index=True)
    # Pivot with two index levels: product_name and sku
    combined['product_name'] = combined['product_name'].fillna('')
    combined['products__variants__sku'] = combined['products__variants__sku'].fillna('')

    pivot = combined.pivot_table(
        index=['product_name', 'products__variants__sku'],
        columns='year_month_str',
        values='quantity',
        fill_value=0,
        aggfunc='sum'
    )

    # Sort columns chronologically
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)

    return pivot


comparison_pivots = {}
for channel in all_channels:
    comparison_pivots[channel] = build_comparison_pivot(channel, monthly_data, forecast_df, sku_details)

print("✅ Comparison pivots built (2024 actuals | 2025 actuals | 2026 forecast)")
# Show column range for verification
sample_cols = list(comparison_pivots['TOTAL'].columns)
print(f"Columns: {sample_cols[0]} → {sample_cols[-1]} ({len(sample_cols)} months)")

✅ Comparison pivots built (2024 actuals | 2025 actuals | 2026 forecast)
Columns: 2024-03 → 2026-12 (34 months)


In [43]:
# Create comparison DataFrames and helper structures
comparison_results = []
for channel in all_channels:
    channel_forecasts = forecast_df[forecast_df['channel'] == channel]
    total_channel_forecast = channel_forecasts['forecast_qty'].sum()
    product_type_forecasts = channel_forecasts.groupby('product_type')['forecast_qty'].sum().reset_index()
    total_by_product_type = product_type_forecasts['forecast_qty'].sum()
    total_by_sku = channel_forecasts.groupby('sku')['forecast_qty'].sum().sum()

    comparison_results.append({
        'Channel': channel,
        'A - Total Forecast': round(total_channel_forecast, 0),
        'B - Product Type': round(total_by_product_type, 0),
        'C - Item Level': round(total_by_sku, 0),
        'B vs A Diff': round(total_by_product_type - total_channel_forecast, 2),
        'C vs A Diff': round(total_by_sku - total_channel_forecast, 2)
    })

comparison_df = pd.DataFrame(comparison_results)

# Product type breakdown
product_type_details = []
for channel in all_channels:
    channel_forecasts = forecast_df[forecast_df['channel'] == channel]
    pt_summary = channel_forecasts.groupby('product_type').agg(
        Total_Forecast_Qty=('forecast_qty', 'sum'),
        Num_SKUs=('sku', 'nunique')
    ).reset_index()
    pt_summary.columns = ['Product_Type', 'Total_Forecast_Qty', 'Num_SKUs']
    pt_summary['Channel'] = channel
    pt_summary['Avg_Per_SKU'] = (pt_summary['Total_Forecast_Qty'] / pt_summary['Num_SKUs']).round(1)
    channel_total = pt_summary['Total_Forecast_Qty'].sum()
    pt_summary['Pct_of_Channel'] = ((pt_summary['Total_Forecast_Qty'] / channel_total) * 100).round(1)
    product_type_details.append(pt_summary)

product_type_df = pd.concat(product_type_details, ignore_index=True)
product_type_df = product_type_df[['Channel', 'Product_Type', 'Num_SKUs', 'Total_Forecast_Qty', 'Avg_Per_SKU', 'Pct_of_Channel']]

# Channel summary (FILTERED TO ACTIVE SKUS ONLY)
summary_by_channel = []
# Filter monthly_data to only active SKUs for fair comparison
active_monthly_data = monthly_data[monthly_data['products__variants__sku'].isin(active_skus_from_file)]

for channel in all_channels:
    if channel == 'TOTAL':
        channel_monthly = active_monthly_data.groupby(['year_month_str'])['quantity'].sum().reset_index()
    else:
        channel_monthly = active_monthly_data[active_monthly_data['orders__source'] == channel].groupby(['year_month_str'])['quantity'].sum().reset_index()
    channel_monthly['year'] = channel_monthly['year_month_str'].str[:4]
    yearly = channel_monthly.groupby('year')['quantity'].sum()
    forecast_total = forecast_df[forecast_df['channel'] == channel]['forecast_qty'].sum()
    summary_by_channel.append({
        'Channel': channel,
        '2022_Total': int(yearly.get('2022', 0)),
        '2023_Total': int(yearly.get('2023', 0)),
        '2024_Total': int(yearly.get('2024', 0)),
        '2025_Total': int(yearly.get('2025', 0)),
        '2026_YTD': int(yearly.get('2026', 0)),
        '2026_Forecast': int(forecast_total)
    })

summary_df = pd.DataFrame(summary_by_channel)
summary_df['YoY_Growth'] = ((summary_df['2026_Forecast'] - summary_df['2025_Total']) / summary_df['2025_Total'] * 100).round(1)

print("✅ Summary dataframes ready")

✅ Summary dataframes ready


In [44]:
# ============================================================
# STEP 6: BUILD AGGREGATED VIEWS — per channel, monthly rows
# ============================================================
# Output structure per channel tab:
#   Product Name | SKU | Month | Year 2024 | Year 2025 | Forecast 2026
# Each SKU has 12 rows (one per calendar month Jan–Dec).
# Year 2024 / Year 2025 = sum of actuals for that month across both years.
# Forecast 2026 = actuals through HISTORY_CUTOFF + statistical forecast for remaining months.
# A subtotal row is added per SKU, and a grand total at the bottom.
#
# Also builds the Building Blocks dataset:
#   Product Name | SKU | Month | Raw 2025 | Deseasonalized 2025 | Seasonal Index | Trend Slope | Forecast 2026
# ============================================================

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

def col_letter(n):
    """Convert 1-based column index to spreadsheet letter(s)."""
    result = ''
    while n > 0:
        n, remainder = divmod(n - 1, 26)
        result = chr(65 + remainder) + result
    return result


def build_channel_monthly_view(channel, monthly_data, forecast_df, sku_details):
    """
    Returns a list of row dicts with columns:
      Product Name, SKU, Product_Type, Month (1-12), Month_Name,
      Year_2024, Year_2025, Forecast_2026
    One row per SKU per calendar month. Only SKUs with any data included.
    """
    # --- actuals ---
    if channel == 'TOTAL':
        act = monthly_data.groupby(['year_month_str','products__variants__sku'])['quantity'].sum().reset_index()
    else:
        act = monthly_data[monthly_data['orders__source'] == channel][
            ['year_month_str','products__variants__sku','quantity']].copy()

    # Normalize SKU strings to avoid whitespace/type mismatch
    act['products__variants__sku'] = act['products__variants__sku'].astype(str).str.strip()
    # No merge with sku_details here — product_name/type are fetched from sku_details
    # in the per-SKU loop below. Merging here caused fan-out duplication when sku_details
    # has multiple rows per SKU (e.g. multiple variants), doubling every actuals row.
    act['year']  = act['year_month_str'].str[:4].astype(int)
    act['month'] = act['year_month_str'].str[5:7].astype(int)

    act_pivot = act.groupby(['products__variants__sku', 'year', 'month'])[
        'quantity'].sum().reset_index()

    # --- forecasts ---
    fcst = forecast_df[forecast_df['channel'] == channel][
        ['sku','product_name','product_type','month','forecast_qty']].copy()
    fcst['sku'] = fcst['sku'].astype(str).str.strip()
    fcst['month_num'] = fcst['month'].str[5:7].astype(int)

    # Pre-compute month numbers on forecast_df once to avoid repeated .str slice in the loop
    if '_month_num' not in forecast_df.columns:
        forecast_df['_month_num'] = forecast_df['month'].str[5:7].astype(int)

    # --- build rows ---
    rows = []
    # Iterate over SKUs that have EITHER historical data OR forecast data
    skus_with_actuals = set(act_pivot['products__variants__sku'].astype(str).str.strip().unique())
    skus_with_forecast = set(fcst['sku'].astype(str).str.strip().unique())
    all_skus = list(skus_with_actuals | skus_with_forecast)

    for sku in all_skus:
        sku_info = sku_details[sku_details['products__variants__sku'] == sku]
        if len(sku_info) == 0:
            continue
        sku_info = sku_info.iloc[0]
        prod_name  = sku_info['product_name']
        prod_type  = sku_info['products__product_type']

        sku_act  = act_pivot[act_pivot['products__variants__sku'] == sku]
        sku_fcst = fcst[fcst['sku'] == sku]

        # Check if this SKU has any data for this channel at all
        # Use .isin() to handle both int and np.int64 types
        sku_2024 = sku_act[sku_act['year'].isin([2024])]
        sku_2025 = sku_act[sku_act['year'].isin([2025])]
        _cutoff_month_num = int(HISTORY_CUTOFF[5:7])  # e.g. 2 when HISTORY_CUTOFF='2026-02'
        sku_2026_act = sku_act[sku_act['year'].isin([2026])]
        has_data = (len(sku_2024) + len(sku_2025) + len(sku_2026_act) + len(sku_fcst)) > 0
        if not has_data:
            continue

        for m in range(1, 13):
            val_2024 = int(sku_2024[sku_2024['month'] == m]['quantity'].sum())
            val_2025 = int(sku_2025[sku_2025['month'] == m]['quantity'].sum())

            # 2026: months up to HISTORY_CUTOFF = actuals, rest = forecast
            if m <= _cutoff_month_num:
                val_2026 = int(sku_2026_act[sku_2026_act['month'] == m]['quantity'].sum())
            else:
                match = sku_fcst[sku_fcst['month_num'] == m]
                if len(match) > 0:
                    val_2026 = int(match['forecast_qty'].sum())
                elif channel == 'TOTAL':
                    # Fall back: sum DTC + Wholesale forecast for this SKU + month
                    dtc_match = forecast_df[
                        (forecast_df['channel'] == 'Direct-to-Consumer') &
                        (forecast_df['sku'].astype(str).str.strip() == sku) &
                        (forecast_df['_month_num'] == m)
                    ]
                    ws_match = forecast_df[
                        (forecast_df['channel'] == 'Wholesale') &
                        (forecast_df['sku'].astype(str).str.strip() == sku) &
                        (forecast_df['_month_num'] == m)
                    ]
                    val_2026 = int(dtc_match['forecast_qty'].sum()) + int(ws_match['forecast_qty'].sum())
                else:
                    val_2026 = 0

            is_active = sku in active_skus_from_file
            pareto_cat = pareto_lookup.get(sku, 'N/A')

            rows.append({
                'Product Name': prod_name,
                'SKU': sku,
                'Product_Type': prod_type,
                'Is_Active': 'Yes' if is_active else 'No',
                'Pareto': pareto_cat,
                'Month': m,
                'Month_Name': MONTHS[m-1],
                'Year_2024': val_2024,
                'Year_2025': val_2025,
                'Forecast_2026': val_2026,
            })

    return pd.DataFrame(rows)


def build_building_blocks(monthly_data, forecast_df, sku_details, pt_seasonal_indices, channel_growth_rates=None):
    """
    For each active SKU, for each calendar month, shows:
      Product Name | SKU | Product_Type | Month | Month_Name |
      Raw_2025 | Seasonal_Index | Deseasonalized_2025 |
      Trend_Slope_Per_Month | Base_Level | Forecast_2026
    """
    # TOTAL channel actuals
    act = monthly_data.groupby(['year_month_str','products__variants__sku'])['quantity'].sum().reset_index()
    act['year']  = act['year_month_str'].str[:4].astype(int)
    act['month'] = act['year_month_str'].str[5:7].astype(int)
    act_2025 = act[act['year'] == 2025].copy()

    # forecasts (TOTAL channel)
    fcst = forecast_df[forecast_df['channel'] == 'TOTAL'][
        ['sku','month','forecast_qty']].copy()
    fcst['month_num'] = fcst['month'].str[5:7].astype(int)

    if channel_growth_rates is None:
        channel_growth_rates = {}

    rows = []
    for sku in sku_details['products__variants__sku'].unique():

        sku_info = sku_details[sku_details['products__variants__sku'] == sku]
        if len(sku_info) == 0:
            continue
        sku_info  = sku_info.iloc[0]
        prod_name = sku_info['product_name']
        prod_type = sku_info['products__product_type']
        seas_idx  = pt_seasonal_indices.get(prod_type, {m: 1.0 for m in range(1,13)})

        # Get historical values — proxy fills pre-launch gaps same as forecast engine
        hist_vals, hist_months = build_history_with_proxy(
            sku, act, proxy_map, HISTORY_CUTOFF
        )

        n = len(hist_vals)
        window = min(n, BASE_LEVEL_WINDOW)
        if window >= 4:
            recent_vals   = hist_vals[-window:]
            recent_months = hist_months[-window:]
            deseason_recent = np.array([
                v / seas_idx.get(int(m), 1.0) if seas_idx.get(int(m), 1.0) > 0 else v
                for v, m in zip(recent_vals, recent_months)
            ])
            base_level    = compute_base_level(deseason_recent, method=BASE_LEVEL_METHOD)
            active_months = int(np.sum(deseason_recent > 0))
            # Mirror forecast engine exactly
            if active_months < MIN_MONTHS_FOR_TREND:
                trend_slope = 0.0
            else:
                trend_slope = calculate_weighted_trend(deseason_recent)
                if base_level > 0:
                    pareto_cat_bb = pareto_lookup.get(sku, 'Top Performers')
                    cap_pct   = TREND_SLOPE_CAP_PCT_BOTTOM20 if pareto_cat_bb == 'Bottom 20%' else TREND_SLOPE_CAP_PCT
                    max_slope = base_level * cap_pct
                    trend_slope = float(np.clip(trend_slope, -max_slope, max_slope))
        else:
            base_level  = float(np.mean(hist_vals)) if n > 0 else 0.0
            trend_slope = 0.0

        sku_2025_act = act_2025[act_2025['products__variants__sku'] == sku]

        _cutoff_year   = int(HISTORY_CUTOFF[:4])
        _cutoff_mo     = int(HISTORY_CUTOFF[5:7])
        base_floor     = base_level * 0.20
        monthly_growth = (1 + CHANNEL_GROWTH_RATES.get('TOTAL', 0.0)) ** (1 / 12) - 1

        # Map each forecast calendar month to its 1-based step index
        _fcst_month_to_step = {cal_m: (i + 1) for i, cal_m in enumerate(forecast_cal_months)}

        for m in range(1, 13):
            raw_2025    = int(sku_2025_act[sku_2025_act['month'] == m]['quantity'].sum())
            si          = round(seas_idx.get(m, 1.0), 4)
            deseas_2025 = round(raw_2025 / si, 1) if si > 0 else raw_2025

            # Step = 0 for actual months, 1-based for forecast months
            step = _fcst_month_to_step.get(m, 0)

            if m <= _cutoff_mo:
                # Show real actuals for months up to cutoff
                actual_val = act[(act['products__variants__sku'] == sku) &
                                 (act['year'] == _cutoff_year) &
                                 (act['month'] == m)]['quantity'].sum()
                fcst_2026 = int(actual_val)
            else:
                # Recompute from building block components — identical formula
                # to calculate_forecast so BB and forecast sheet always agree
                projected_base  = max(base_level + trend_slope * step, base_floor)
                projected_base *= (1 + monthly_growth) ** step
                fcst_2026       = int(round(max(0, projected_base * si)))

            pareto_cat = pareto_lookup.get(sku, 'N/A')

            rows.append({
                'Product Name': prod_name,
                'SKU': sku,
                'Product_Type': prod_type,
                'Pareto': pareto_cat,
                'Month': m,
                'Month_Name': MONTHS[m-1],
                'Step': step,
                'Raw_2025_Actual': raw_2025,
                'Seasonal_Index': si,
                'Deseasonalized_2025': deseas_2025,
                'Base_Level_(deseason)': round(base_level, 1),
                'Trend_Slope_(units/mo)': round(trend_slope, 4),
                'Growth_Rate_Override': {ch: f'{r*100:+.1f}%' for ch, r in channel_growth_rates.items()}.get('TOTAL', '+0.0%'),
                'DTC_Growth_Override':  f"{channel_growth_rates.get('Direct-to-Consumer', 0.0)*100:+.1f}%",
                'Wholesale_Growth_Override': f"{channel_growth_rates.get('Wholesale', 0.0)*100:+.1f}%",
                'Total_Growth_Override': f"{channel_growth_rates.get('TOTAL', 0.0)*100:+.1f}%",
                'Forecast_2026': fcst_2026,
            })

    return pd.DataFrame(rows)


# Build all views
print("Building per-channel monthly views...")
channel_monthly_views = {}
for ch in all_channels:
    channel_monthly_views[ch] = build_channel_monthly_view(
        ch, monthly_data, forecast_df, sku_details)
    n_skus = channel_monthly_views[ch]['SKU'].nunique()
    print(f"  {ch}: {n_skus} SKUs × 12 months = {len(channel_monthly_views[ch])} rows")

print("\nBuilding building blocks view...")
building_blocks_df = build_building_blocks(
    monthly_data, forecast_df, sku_details, pt_seasonal_indices,
    channel_growth_rates=CHANNEL_GROWTH_RATES)
print(f"  Building Blocks: {len(building_blocks_df)} rows ({building_blocks_df['SKU'].nunique()} active SKUs)")
print("\n✅ All views ready")

Building per-channel monthly views...
  Direct-to-Consumer: 220 SKUs × 12 months = 2640 rows
  Wholesale: 150 SKUs × 12 months = 1800 rows
  TOTAL: 277 SKUs × 12 months = 3324 rows

Building building blocks view...
  Building Blocks: 3324 rows (277 active SKUs)

✅ All views ready


In [45]:
# ============================================================
# BUILD: Aggregated By Year (for QC comparison)
# ============================================================

agg_by_year = []

# Aggregate monthly_data to yearly totals per SKU
total_monthly = monthly_data.groupby(['year_month_str', 'products__variants__sku'])['quantity'].sum().reset_index()
total_monthly['year'] = total_monthly['year_month_str'].str[:4]

for sku in sku_details['products__variants__sku'].unique():
    sku_info = sku_details[sku_details['products__variants__sku'] == sku].iloc[0]
    sku_data = total_monthly[total_monthly['products__variants__sku'] == sku]

    yearly = sku_data.groupby('year')['quantity'].sum()

    # 2026: Jan YTD actual
    ytd_2026 = int(sku_data[sku_data['year_month_str'] == HISTORY_CUTOFF]['quantity'].sum())

    # 2026 Forecast: sum from forecast_df
    sku_fcst_2026 = forecast_df[
        (forecast_df['sku'] == sku) & (forecast_df['channel'] == 'TOTAL')
    ]['forecast_qty'].sum()

    total_2026 = ytd_2026 + int(sku_fcst_2026)

    is_active = sku in active_skus_from_file

    agg_by_year.append({
        'SKU': sku,
        'Product_Name': sku_info['product_name'],
        'Is_Active': 'Yes' if is_active else 'No',
        '2024': int(yearly.get('2024', 0)),
        '2025': int(yearly.get('2025', 0)),
        '2026_YTD': ytd_2026,
        '2026_Forecast': total_2026
    })

agg_by_year_df = pd.DataFrame(agg_by_year)
print(f"\n✅ Aggregated By Year built: {len(agg_by_year_df)} SKUs")
print(f"   Columns: {list(agg_by_year_df.columns)}")



✅ Aggregated By Year built: 277 SKUs
   Columns: ['SKU', 'Product_Name', 'Is_Active', '2024', '2025', '2026_YTD', '2026_Forecast']


In [46]:
# Authenticate with Google and create new Google Sheet
import gspread
from google.colab import auth
from google.auth import default

print("Authenticating with Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Authenticated")

Authenticating with Google...
✅ Authenticated


In [47]:
# Create new Google Sheet
sheet_name = f"Demand_Planning_{datetime.now().strftime('%Y%m%d_%H%M')}"
sh = gc.create(sheet_name)
print(f"Created Google Sheet: {sheet_name}")
print(f"URL: https://docs.google.com/spreadsheets/d/{sh.id}")

Created Google Sheet: Demand_Planning_20260309_1758
URL: https://docs.google.com/spreadsheets/d/1GmyyJ8fgq6OXerlYYfyEBMu-isQ2uY6Xiyq3POMafLA


In [48]:
# ============================================================
# EXPORT TAB: Flat export format for downstream systems
# ============================================================

import calendar

print("Building Export Tab data...")

# ── 0. Build SKU → products__variants__title lookup ──────────────────────
sku_title_map = (
    df[['products__variants__sku', 'products__variants__title']]
    .dropna(subset=['products__variants__sku', 'products__variants__title'])
    .drop_duplicates(subset=['products__variants__sku'])
    .set_index('products__variants__sku')['products__variants__title']
    .to_dict()
)
print(f"  SKU→title lookup built: {len(sku_title_map)} entries")

# ── 1. Build customer_category mix from raw data ─────────────────────────
if 'customers__customer_category' in df.columns:
    print("  Found customers__customer_category column")
    raw_2025 = df[
        (df['created_date'].dt.year == 2025) &
        (df['products__variants__sku'].isin(active_skus_from_file))
    ].copy()
    raw_2025['channel'] = raw_2025['orders__source']
    mix = raw_2025.groupby([
        'products__variants__sku', 'channel', 'customers__customer_category'
    ])['quantity'].sum().reset_index()
    mix.columns = ['sku', 'channel', 'customer_category', 'qty']
    mix_total = mix.groupby(['sku', 'channel'])['qty'].sum().reset_index()
    mix_total.columns = ['sku', 'channel', 'total_qty']
    mix = mix.merge(mix_total, on=['sku', 'channel'], how='left')
    mix['share'] = mix['qty'] / mix['total_qty']
    use_category = True
    print(f"  Customer categories found: {sorted(mix['customer_category'].unique())}")
else:
    print("  ⚠️  No customers__customer_category column — exporting without category split")
    use_category = False

# ── Derive actual months from HISTORY_CUTOFF ─────────────────────────────
_cutoff_year = int(HISTORY_CUTOFF[:4])   # e.g. 2026
_cutoff_mo   = int(HISTORY_CUTOFF[5:7])  # e.g. 2 — so actuals = Jan + Feb
actual_months = list(range(1, _cutoff_mo + 1))  # [1, 2] when cutoff is Feb

export_channels = ['Direct-to-Consumer', 'Wholesale']

MONTH_LABELS = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
    9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

export_rows = []

# ── 2. Build actuals rows for all months up to HISTORY_CUTOFF ────────────
raw_actuals = df[
    (df['created_date'].dt.year == _cutoff_year) &
    (df['created_date'].dt.month.isin(actual_months)) &
    (df['products__variants__sku'].isin(active_skus_from_file)) &
    (df['orders__source'].isin(export_channels))
].copy()
raw_actuals['channel'] = raw_actuals['orders__source']
raw_actuals['month']   = raw_actuals['created_date'].dt.month

if use_category:
    actuals_agg = raw_actuals.groupby([
        'products__variants__sku', 'channel', 'month', 'customers__customer_category'
    ])['quantity'].sum().reset_index()
    actuals_agg.columns = ['sku', 'channel', 'month', 'customer_category', 'units']
else:
    actuals_agg = raw_actuals.groupby([
        'products__variants__sku', 'channel', 'month'
    ])['quantity'].sum().reset_index()
    actuals_agg.columns = ['sku', 'channel', 'month', 'units']
    actuals_agg['customer_category'] = ''

for _, row in actuals_agg.iterrows():
    sku        = row['sku']
    channel    = row['channel']
    mo         = int(row['month'])
    cat        = row['customer_category']
    cat_units  = int(round(row['units']))
    prod_name  = sku_title_map.get(sku, sku)
    full_month      = calendar.month_name[mo]
    month_year_full = f"{full_month} {_cutoff_year}"
    month_abbr      = MONTH_LABELS[mo]
    composite_key   = f"{prod_name}-{sku}-{channel}-{month_year_full}"

    export_rows.append({
        'composite_primary_key':        composite_key,
        'product_name':                  prod_name,
        'code':                          sku,
        'channel':                       channel,
        'customers__customer_category':  cat,
        'month_num':                     f"{_cutoff_year}{mo:02d}",
        'month_year':                    f"{month_abbr}-{str(_cutoff_year)[2:]}",
        'units':                         cat_units,
    })

print(f"  {_cutoff_year} actuals rows ({MONTH_LABELS[1]}–{MONTH_LABELS[_cutoff_mo]}): {len(export_rows)}")

# ── 3. Build forecast rows (FORECAST_START onward only — no actuals overlap) ──
forecast_export = forecast_df[
    (forecast_df['channel'].isin(export_channels)) &
    (forecast_df['month'] >= FORECAST_START)   # excludes any actual months
].copy()

for _, fcst_row in forecast_export.iterrows():
    sku       = fcst_row['sku']
    channel   = fcst_row['channel']
    month_str = fcst_row['month']
    units     = fcst_row['forecast_qty']

    prod_name = sku_title_map.get(sku, fcst_row.get('product_name', sku))

    year_num   = int(month_str[:4])
    month_num  = int(month_str[5:7])
    month_abbr = MONTH_LABELS[month_num]
    full_month      = calendar.month_name[month_num]
    month_year_full = f"{full_month} {year_num}"

    if use_category:
        sku_mix  = mix[(mix['sku'] == sku) & (mix['channel'] == channel)]
        cat_rows = [('Unknown', 1.0)] if len(sku_mix) == 0 else list(zip(sku_mix['customer_category'], sku_mix['share']))
    else:
        cat_rows = [('', 1.0)]

    for cat, share in cat_rows:
        cat_units     = int(round(units * share))
        composite_key = f"{prod_name}-{sku}-{channel}-{month_year_full}"

        export_rows.append({
            'composite_primary_key':        composite_key,
            'product_name':                  prod_name,
            'code':                          sku,
            'channel':                       channel,
            'customers__customer_category':  cat,
            'month_num':                     f"{year_num}{month_num:02d}",
            'month_year':                    f"{month_abbr}-{str(year_num)[2:]}",
            'units':                         cat_units,
        })

export_df = pd.DataFrame(export_rows)
export_df = export_df.sort_values(['code', 'channel', 'month_num', 'customers__customer_category']).reset_index(drop=True)

print(f"\n✅ Export data built: {len(export_df):,} rows")
print(f"   Channels: {export_df['channel'].unique()}")
print(f"   Months: {sorted(export_df['month_year'].unique())}")
if len(export_df) > 0:
    print(f"\nSample rows:")
    display(export_df.head(5))

# ── 4. Write to Google Sheet ──────────────────────────────────────────────
print("\nWriting Export tab to Google Sheet...")

ws_export = sh.add_worksheet(title='Export', rows=len(export_df)+10, cols=10)

header = ['composite_primary_key', 'product_name', 'code', 'channel',
          'customers__customer_category', 'month_num', 'month_year', 'units']

data_rows = [header] + export_df[header].values.tolist()
ws_export.update('A1', data_rows)

ws_export.format('A1:H1', {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.18, 'green': 0.33, 'blue': 0.53}
})
ws_export.freeze(rows=1)

print(f"  ✅ Export tab written: {len(data_rows)} rows (including header)")
print(f"     Google Sheet URL: https://docs.google.com/spreadsheets/d/{sh.id}")

Building Export Tab data...
  SKU→title lookup built: 277 entries
  Found customers__customer_category column
  Customer categories found: ['Direct-to-Consumer', 'No Customer Type', 'Professional - Esthetician', 'Retailer - Credo', 'Retailer - Non-Credo']
  2026 actuals rows (Jan–Feb): 272

✅ Export data built: 4,412 rows
   Channels: ['Wholesale' 'Direct-to-Consumer']
   Months: ['Apr-26', 'Aug-26', 'Dec-26', 'Feb-26', 'Jan-26', 'Jul-26', 'Jun-26', 'Mar-26', 'May-26', 'Nov-26', 'Oct-26', 'Sep-26']

Sample rows:


,composite_primary_key,product_name,code,channel,customers__customer_category,month_num,month_year,units
0,Product Variant 0231-SKU-004F49D5-Wholesale-Ma...,Product Variant 0231,SKU-004F49D5,Wholesale,Professional - Esthetician,202603,Mar-26,1
1,Product Variant 0231-SKU-004F49D5-Wholesale-Ap...,Product Variant 0231,SKU-004F49D5,Wholesale,Professional - Esthetician,202604,Apr-26,2
2,Product Variant 0231-SKU-004F49D5-Wholesale-Ma...,Product Variant 0231,SKU-004F49D5,Wholesale,Professional - Esthetician,202605,May-26,1
3,Product Variant 0231-SKU-004F49D5-Wholesale-Ju...,Product Variant 0231,SKU-004F49D5,Wholesale,Professional - Esthetician,202606,Jun-26,2
4,Product Variant 0231-SKU-004F49D5-Wholesale-Ju...,Product Variant 0231,SKU-004F49D5,Wholesale,Professional - Esthetician,202607,Jul-26,1



Writing Export tab to Google Sheet...
  ✅ Export tab written: 4413 rows (including header)
     Google Sheet URL: https://docs.google.com/spreadsheets/d/1GmyyJ8fgq6OXerlYYfyEBMu-isQ2uY6Xiyq3POMafLA


In [49]:
# ============================================================
# WRITE: Summary Dashboard
# ============================================================
summary_sheet = sh.sheet1
summary_sheet.update_title('Summary Dashboard')

summary_sheet.update('A1', [['DEMAND PLANNING SUMMARY']])
summary_sheet.update('A3', [[f'Historical Period: through {HISTORY_CUTOFF}']])
summary_sheet.update('A4', [[f'Forecast Period: {forecast_months[0]} through {forecast_months[-1]}']])

# --- Methodology explanation ---
method_rows = [
    ['FORECASTING METHODOLOGY'],
    [''],
    ['SEASONALITY (Product-Type Level)'],
    ['  Seasonal indices are calculated by pooling all SKUs within each product type together, using the last 24 months'],
    ['  of TOTAL channel sales data. For each calendar month (Jan–Dec), we compute the average volume relative to the'],
    ['  overall monthly average for that product type. This produces a stable index (e.g. 1.42 = 42% above average,'],
    ['  0.71 = 29% below average) that is shared by all SKUs in the same product type. Using product-type pooling'],
    ['  rather than per-item indices prevents noisy, low-volume SKUs from generating unreliable seasonal patterns.'],
    [''],
    ['TREND (Per-SKU, Recency-Weighted)'],
    ['  A linear trend is estimated for each SKU individually using the last 12 months of deseasonalized sales.'],
    ['  Weighted least-squares regression is used, where the most recent month carries 12x the weight of the'],
    ['  oldest month in the window. This means recent acceleration or deceleration in demand has a much stronger'],
    ['  influence on the slope than older data. The trend slope is capped at ±2.5% of the base level per month'],
    ['  (~30% annually) to prevent runaway projections on sparse or erratic SKUs.'],
    [''],
    ['BASE LEVEL (Per-SKU)'],
    [f'  Computed using BASE_LEVEL_METHOD={BASE_LEVEL_METHOD!r} over the last {BASE_LEVEL_WINDOW} months of deseasonalized history.'],
    ['  Methods: mean (default, most stable), median (outlier-robust), ewm (recency-weighted).'],
    ['  Change BASE_LEVEL_METHOD, BASE_LEVEL_WINDOW, BASE_LEVEL_ALPHA in the config cell.'],
    [''],
    ['FORECAST CALCULATION'],
    ['  For each future month: Forecast = (Base Level + Trend × Steps Ahead) × Seasonal Index × Growth Factor'],
    ['  Seasonal Index: product-type index for that calendar month'],
    ['  Growth Factor: optional manual override (default 0% = pure statistical forecast)'],
    ['  All forecasts are rounded to whole units — no fractional quantities.'],
    [''],
    ['INACTIVITY RULE'],
    ['  Any SKU with zero total sales across all channels in the 6-month window prior to the forecast start'],
    ['  is classified as inactive and receives no forecast. These SKUs still appear in historical views.'],
    [''],
    ['WHAT THIS FORECAST DOES NOT CAPTURE'],
    ['  The statistical model extrapolates demand patterns from historical sell-through data. It does not'],
    ['  account for the following factors, which should be applied as manual judgment on top of the forecast:'],
    ['  • Promotions & Discounts: Heavy discounting (e.g. sitewide sales) inflates historical volume in'],
    ['    those months. The model will partially absorb this into the base level and seasonal index,'],
    ['    which can cause future months to be over- or under-forecast relative to promo intent.'],
    ['  • New Product Launches: SKUs with < 6 months of history have limited trend signal. Review'],
    ['    their forecasts manually and consider applying a growth override.'],
    ['  • Planned Price Changes: A price increase typically suppresses volume; a reduction lifts it.'],
    ['    Neither is visible to the model.'],
    ['  • Inventory / Supply Constraints: Stockouts in the historical window appear as zero demand,'],
    ['    causing the model to underestimate true underlying demand for those periods.'],
    ['  • Channel Mix Shifts: If volume is intentionally being moved between DTC and Wholesale,'],
    ['    the channel-level forecasts will not reflect that intent.'],
    ['  • Discontinued SKUs: Captured by the inactivity rule but only if sales went to zero in the'],
    ['    last 6 months. SKUs being wound down gradually will still receive a (likely too-high) forecast.'],
    [''],
]
summary_sheet.update('A6', method_rows)

method_end_row = 6 + len(method_rows) + 1
summary_sheet.update(f'A{method_end_row}', [['CHANNEL SUMMARY']])
summary_data = [summary_df.columns.tolist()] + summary_df.values.tolist()
summary_sheet.update(f'A{method_end_row + 2}', summary_data)

comparison_start = method_end_row + 12
summary_sheet.update(f'A{comparison_start}', [['FORECAST AGGREGATION COMPARISON']])
comparison_data = [comparison_df.columns.tolist()] + comparison_df.values.tolist()
summary_sheet.update(f'A{comparison_start + 2}', comparison_data)

summary_sheet.format('A1', {'textFormat': {'bold': True, 'fontSize': 14}})
summary_sheet.format('A6', {'textFormat': {'bold': True, 'fontSize': 12}})
summary_sheet.format('A8', {'textFormat': {'bold': True, 'underline': True}})
summary_sheet.format('A14', {'textFormat': {'bold': True, 'underline': True}})
summary_sheet.format('A21', {'textFormat': {'bold': True, 'underline': True}})
summary_sheet.format('A27', {'textFormat': {'bold': True, 'underline': True}})
summary_sheet.format('A33', {'textFormat': {'bold': True, 'underline': True}})
summary_sheet.format(f'A{method_end_row}', {'textFormat': {'bold': True, 'fontSize': 12}})
summary_sheet.format(f'A{method_end_row + 2}:H{method_end_row + 2}', {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
})
summary_sheet.format(f'A{comparison_start}', {'textFormat': {'bold': True, 'fontSize': 12}})
summary_sheet.format(f'A{comparison_start + 2}:F{comparison_start + 2}', {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
})

print("Summary Dashboard created with full methodology explanation")

Summary Dashboard created with full methodology explanation


In [50]:
# ============================================================
# WRITE: Product Type Breakdown (with Seasonality Table)
# ============================================================
pt_sheet = sh.add_worksheet(title='Product Type Breakdown', rows=1000, cols=30)

# Section 1: Seasonality indices
pt_sheet.update('A1', [['SEASONAL INDICES BY PRODUCT TYPE AND MONTH']])
pt_sheet.update('A2', [['Based on last 24 months of TOTAL channel data. Index > 1.0 = stronger than average. Normalized so each row averages to 1.0.']])

seas_data = [seasonality_display_df.columns.tolist()] + seasonality_display_df.values.tolist()
pt_sheet.update('A4', seas_data)

# Header formatting for seasonality table
pt_sheet.format('A1', {'textFormat': {'bold': True, 'fontSize': 13}})
num_pt_rows = len(seasonality_display_df)
header_range = f'A4:M4'
pt_sheet.format(header_range, {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.18, 'green': 0.53, 'blue': 0.33}  # green
})

# Section 2: Forecast summary by product type
sep_row = num_pt_rows + 7  # leave a gap
pt_sheet.update(f'A{sep_row}', [['FORECAST SUMMARY BY PRODUCT TYPE AND CHANNEL']])
pt_sheet.format(f'A{sep_row}', {'textFormat': {'bold': True, 'fontSize': 13}})

pt_data = [product_type_df.columns.tolist()] + product_type_df.values.tolist()
pt_sheet.update(f'A{sep_row + 2}', pt_data)
pt_sheet.format(f'A{sep_row + 2}:F{sep_row + 2}', {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
})

print("Product Type Breakdown sheet created (with seasonality indices)")

Product Type Breakdown sheet created (with seasonality indices)


In [51]:
# ============================================================
# WRITE: Channel Forecast + Comparison Pivot sheets
# Each channel gets ONE sheet: 2024 actuals | 2025 actuals | 2026 (Jan actual + Feb-Dec forecast)
# ============================================================

# FORECAST_START is derived from HISTORY_CUTOFF in the config cell above

import time # Import time for delays

for channel in all_channels:
    print(f"Creating forecast sheet for {channel}...")

    sheet_title = f"{channel} - Forecast"
    # Check if sheet already exists and delete it
    try:
        existing_ws = sh.worksheet(sheet_title)
        sh.del_worksheet(existing_ws)
        print(f"  Deleted existing sheet: {sheet_title}")
        time.sleep(1) # Small delay after deleting a sheet
    except gspread.exceptions.WorksheetNotFound:
        pass # Sheet does not exist, proceed to create

    ws = sh.add_worksheet(title=sheet_title, rows=2000, cols=150)
    # The previous time.sleep(5) was here, but it's not sufficient given the many subsequent updates.
    # It will be replaced by a longer sleep at the end of the loop iteration for each channel.

    pivot = comparison_pivots[channel]
    all_cols = list(pivot.columns)  # chronologically sorted month strings

    # Identify which columns are forecast vs actual
    # Anything before FORECAST_START = actual; FORECAST_START onward = forecast
    actual_cols = [c for c in all_cols if c < FORECAST_START]
    forecast_cols = [c for c in all_cols if c >= FORECAST_START]

    # Build header rows
    # Row 1: title
    ws.update('A1', [[f'{channel} — Historical vs Forecast']])

    # Row 2: year group labels  (2024 actuals / 2025 actuals / 2026 actual+forecast)
    col_labels = []
    for c in all_cols:
        yr = c[:4]
        is_fcst = (c >= FORECAST_START)
        label = f"{yr} {'[FORECAST]' if is_fcst else '[ACTUAL]'}"
        col_labels.append(label)
    year_label_row = ['Product Name', 'SKU'] + col_labels

    # Row 3: month column headers
    month_header_row = ['Product Name', 'SKU'] + all_cols

    # Data rows — MultiIndex: (product_name, sku)
    data_rows = []
    for idx in pivot.index:
        prod_name, sku_code = idx
        row = [prod_name, sku_code] + [int(round(pivot.loc[idx, c])) if c in pivot.columns else 0 for c in all_cols]
        data_rows.append(row)

    # Add a TOTALS row
    totals_row_data = ['** CHANNEL TOTAL **', '']
    for c in all_cols:
        if c in pivot.columns:
            totals_row_data.append(int(round(pivot[c].sum())))
        else:
            totals_row_data.append(0)
    data_rows.append(totals_row_data)

    ws.update('A2', [year_label_row])
    ws.update('A3', [month_header_row])
    ws.update('A4', data_rows)

    # Format headers
    ws.format('A1', {'textFormat': {'bold': True, 'fontSize': 13}})

    # Row 2: alternating year/type shading
    # Row 3: column month headers — bold
    ws.format('A3', {'textFormat': {'bold': True}})

    # Color the actual columns header (row 2) in grey-blue
    num_actual = len(actual_cols)
    num_forecast = len(forecast_cols)

    if num_actual > 0:
        # col_letter() is defined in cell 15
        actual_start_col = col_letter(2)  # B
        actual_end_col = col_letter(1 + num_actual)
        ws.format(f'{actual_start_col}2:{actual_end_col}2', {
            'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
            'backgroundColor': {'red': 0.36, 'green': 0.44, 'blue': 0.56}  # slate
        })
        ws.format(f'{actual_start_col}3:{actual_end_col}3', {
            'textFormat': {'bold': True}
        })

    if num_forecast > 0:
        fcst_start_col = col_letter(2 + num_actual)
        fcst_end_col = col_letter(1 + num_actual + num_forecast)
        ws.format(f'{fcst_start_col}2:{fcst_end_col}2', {
            'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
            'backgroundColor': {'red': 0.18, 'green': 0.53, 'blue': 0.33}  # green for forecast
        })
        ws.format(f'{fcst_start_col}3:{fcst_end_col}3', {
            'textFormat': {'bold': True, 'foregroundColor': {'red': 0.1, 'green': 0.5, 'blue': 0.2}}
        })

    print(f"  → {len(all_cols)} months ({num_actual} actuals + {num_forecast} forecast)")
    time.sleep(30) # Wait for 30 seconds after all operations for a single channel are complete
                  # before starting the next channel.

print("\nAll channel forecast sheets created")

Creating forecast sheet for Direct-to-Consumer...
  → 34 months (24 actuals + 10 forecast)
Creating forecast sheet for Wholesale...
  → 34 months (24 actuals + 10 forecast)
Creating forecast sheet for TOTAL...
  → 34 months (24 actuals + 10 forecast)

All channel forecast sheets created


In [52]:
# ============================================================
# WRITE: Aggregated By Channel (3 tabs) + Building Blocks
# ============================================================

import time

def safe_add_worksheet(sh, title, rows, cols):
    """Delete and recreate a sheet — handles re-runs without duplicate name errors."""
    existing = [ws for ws in sh.worksheets() if ws.title == title]
    if existing:
        sh.del_worksheet(existing[0])
        time.sleep(3)
    return sh.add_worksheet(title=title, rows=rows, cols=cols)

def write_channel_agg_sheet(sh, title, df):
    """
    Write a channel aggregated sheet.
    Layout: Product Name | SKU | Month | Year 2024 | Year 2025 | Forecast 2026
    Each SKU spans 12 rows (one per month) followed by a subtotal row.
    Grand total at the bottom.
    """
    ws = sh.add_worksheet(title=title, rows=5000, cols=10)

    header = ['Product Name', 'SKU', 'Product Type', 'Is Active', 'Pareto', 'Month', 'Year 2024', 'Year 2025', 'Forecast 2026']
    data_rows = [header]

    skus_in_order = df['SKU'].unique()
    grand = {'Year_2024': 0, 'Year_2025': 0, 'Forecast_2026': 0}

    for sku in skus_in_order:
        sku_df = df[df['SKU'] == sku].sort_values('Month')
        prod_name = sku_df['Product Name'].iloc[0]
        prod_type = str(sku_df['Product_Type'].iloc[0]) if pd.notna(sku_df['Product_Type'].iloc[0]) else ''
        is_active = sku_df['Is_Active'].iloc[0]
        pareto = sku_df['Pareto'].iloc[0]
        tot_2024 = sku_df['Year_2024'].sum()
        tot_2025 = sku_df['Year_2025'].sum()
        tot_2026 = sku_df['Forecast_2026'].sum()

        for _, row in sku_df.iterrows():
            data_rows.append([
                prod_name, sku, prod_type, is_active, pareto,
                row['Month_Name'],
                row['Year_2024'], row['Year_2025'], row['Forecast_2026']
            ])

        # Subtotal row for this SKU
        data_rows.append([prod_name + ' — TOTAL', sku, prod_type, is_active, pareto, 'ANNUAL',
                          int(tot_2024), int(tot_2025), int(tot_2026)])
        data_rows.append(['', '', '', '', '', '', '', '', ''])  # spacer

        grand['Year_2024']     += tot_2024
        grand['Year_2025']     += tot_2025
        grand['Forecast_2026'] += tot_2026

    # Grand total
    data_rows.append(['** GRAND TOTAL **', '', '', '', '', '',
                      int(grand["Year_2024"]), int(grand["Year_2025"]), int(grand["Forecast_2026"])])

    ws.update('A1', data_rows)

    # Formatting
    num_rows = len(data_rows)
    ws.format('A1:I1', {
        'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
        'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
    })
    ws.format(f'A{num_rows}:I{num_rows}', {
        'textFormat': {'bold': True},
        'backgroundColor': {'red': 0.95, 'green': 0.95, 'blue': 0.75}
    })

    # Color Year 2024 header grey, Year 2025 slate, Forecast 2026 green
    ws.format('G1', {'backgroundColor': {'red': 0.75, 'green': 0.75, 'blue': 0.75}})
    ws.format('H1', {'backgroundColor': {'red': 0.36, 'green': 0.44, 'blue': 0.56},
                     'textFormat': {'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}, 'bold': True}})
    ws.format('I1', {'backgroundColor': {'red': 0.18, 'green': 0.53, 'blue': 0.33},
                     'textFormat': {'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}, 'bold': True}})

    print(f"  ✅ {title}: {len(data_rows)} rows written")
    return ws


def write_building_blocks_sheet(sh, df):
    # Delete existing sheet if it exists (from a previous partial run)
    existing = [ws for ws in sh.worksheets() if ws.title == 'Building Blocks']
    if existing:
        sh.del_worksheet(existing[0])
    time.sleep(3)
    ws = sh.add_worksheet(title='Building Blocks', rows=5000, cols=16)

    header = [
        'Product Name', 'SKU', 'Product Type', 'Pareto', 'Month #', 'Month',
        'Step', 'Raw 2025 Actual', 'Seasonal Index', 'Deseasonalized 2025',
        'Base Level (deseason)', 'Trend Slope (units/mo)',
        'DTC Growth Override', 'Wholesale Growth Override', 'Total Growth Override',
        'Forecast 2026'
    ]

    data_rows = [header]
    skus = df['SKU'].unique()

    for sku in skus:
        sku_df = df[df['SKU'] == sku].sort_values('Month')
        prod_name = sku_df['Product Name'].iloc[0]
        prod_type = str(sku_df['Product_Type'].iloc[0]) if pd.notna(sku_df['Product_Type'].iloc[0]) else ''

        for _, row in sku_df.iterrows():
            pareto = row['Pareto']
            data_rows.append([
                prod_name, sku, prod_type, pareto,
                int(row['Month']), row['Month_Name'],
                int(row['Step']),
                row['Raw_2025_Actual'],
                row['Seasonal_Index'],
                row['Deseasonalized_2025'],
                row['Base_Level_(deseason)'],
                row['Trend_Slope_(units/mo)'],
                row['DTC_Growth_Override'],
                row['Wholesale_Growth_Override'],
                row['Total_Growth_Override'],
                row['Forecast_2026'],
            ])
        data_rows.append(['', '', '', '', '', '', '', '', '', '', '', ''])  # spacer

    ws.update('A1', data_rows)

    ws.format('A1:P1', {
        'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
        'backgroundColor': {'red': 0.18, 'green': 0.33, 'blue': 0.53}  # deep blue
    })
    # Column colours (shifted one right by new Step col in G)
    ws.format('G1', {'backgroundColor': {'red': 0.36, 'green': 0.44, 'blue': 0.56}})  # step = slate
    ws.format('H1', {'backgroundColor': {'red': 0.36, 'green': 0.44, 'blue': 0.56}})  # raw = slate
    ws.format('I1', {'backgroundColor': {'red': 0.60, 'green': 0.40, 'blue': 0.70}})  # index = purple
    ws.format('J1', {'backgroundColor': {'red': 0.60, 'green': 0.40, 'blue': 0.70}})  # deseas = purple
    ws.format('K1', {'backgroundColor': {'red': 0.85, 'green': 0.65, 'blue': 0.13}})  # base = amber
    ws.format('L1', {'backgroundColor': {'red': 0.85, 'green': 0.65, 'blue': 0.13}})  # trend = amber
    ws.format('M1:O1', {
        'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
        'backgroundColor': {'red': 0.82, 'green': 0.35, 'blue': 0.20}  # burnt orange = override
    })
    ws.format('P1', {'backgroundColor': {'red': 0.18, 'green': 0.53, 'blue': 0.33}})  # forecast = green

    print(f"  ✅ Building Blocks: {len(data_rows)} rows written")
    return ws


# Write the three channel tabs
import time

# Write the three channel tabs
print("Writing Aggregated By Channel sheets...")

for ch in all_channels:
    write_channel_agg_sheet(sh, f"Agg by Month — {ch}", channel_monthly_views[ch])
    time.sleep(15)  # pause between large sheet writes

# Write Building Blocks tab
print("\nWriting Building Blocks sheet...")
write_building_blocks_sheet(sh, building_blocks_df)
time.sleep(15)

# Write Pareto Analysis sheet
print("\nWriting Pareto Analysis sheet...")

# Write Building Blocks tab
print("\nWriting Building Blocks sheet...")
write_building_blocks_sheet(sh, building_blocks_df)

# Write Pareto Analysis sheet
print("\nWriting Pareto Analysis sheet...")
ws_pareto = sh.add_worksheet(title='Pareto Analysis', rows=1000, cols=10)

pareto_header = list(pareto_sheet_data.columns)
# Sanitise all values — Google Sheets API rejects NaN/inf
pareto_rows = [pareto_header] + [
    [_safe(v) for v in row]
    for row in pareto_sheet_data.values.tolist()
]
ws_pareto.update('A1', pareto_rows)

# Format header
ws_pareto.format('A1:H1', {
    'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
    'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
})

# Highlight Top Performers rows in light green — batch all ranges into one API call
top_rows = [
    i + 2 for i, cat in enumerate(pareto_sheet_data['Pareto Category'])
    if cat == 'Top Performers'
]
if top_rows:
    # Build a list of range dicts for a single batch_format call
    pareto_formats = [
        {'range': f'A{r}:H{r}', 'format': {'backgroundColor': {'red': 0.85, 'green': 0.95, 'blue': 0.85}}}
        for r in top_rows
    ]
    ws_pareto.batch_format(pareto_formats)

print(f"  ✅ Pareto Analysis: {len(pareto_rows)} rows written")
print("\n✅ All aggregated sheets created")


Writing Aggregated By Channel sheets...
  ✅ Agg by Month — Direct-to-Consumer: 3082 rows written
  ✅ Agg by Month — Wholesale: 2102 rows written
  ✅ Agg by Month — TOTAL: 3880 rows written

Writing Building Blocks sheet...
  ✅ Building Blocks: 3602 rows written

Writing Pareto Analysis sheet...

Writing Building Blocks sheet...
  ✅ Building Blocks: 3602 rows written

Writing Pareto Analysis sheet...
  ✅ Pareto Analysis: 241 rows written

✅ All aggregated sheets created


In [53]:
# ============================================================
# UPLOAD: QC Check CSV
# ============================================================
# Upload a CSV with columns: finished_good, sales_2024, sales_2025, sales_2026
# This will be compared against the forecast data for quality checking

print("📂 Upload your QC check CSV file")
print("   Expected columns: finished_good, sales_2024, sales_2025, sales_2026")
print()

try:
    qc_files = files.upload()
    if qc_files:
        qc_filename = list(qc_files.keys())[0]
        print(f"\nLoading QC data from: {qc_filename}")

        qc_data = pd.read_csv(qc_filename)

        # Rename columns to match
        qc_data = qc_data.rename(columns={'finished_good': 'SKU'})

        print(f"✅ QC data loaded: {len(qc_data)} SKUs")
        print(f"   Columns: {list(qc_data.columns)}")
        print(f"   Sample SKUs: {qc_data["SKU"].head(3).tolist()}")
    else:
        qc_data = None
        print("⚠️  No QC file uploaded - skipping QC check")
except Exception as e:
    print(f"⚠️  Error loading QC file: {e}")
    qc_data = None


📂 Upload your QC check CSV file
   Expected columns: finished_good, sales_2024, sales_2025, sales_2026



Saving anonymized_sales_by_sku.csv to anonymized_sales_by_sku.csv

Loading QC data from: anonymized_sales_by_sku.csv
✅ QC data loaded: 69 SKUs
   Columns: ['SKU', 'sales_2024', 'sales_2025', 'sales_2026']
   Sample SKUs: ['SKU-31B18550', 'SKU-F9E36644', 'SKU-2EFD6C7A']


In [54]:
# ============================================================
# QC COMPARISON & SHEET CREATION
# ============================================================

if qc_data is not None:
    print("\nBuilding QC comparison...")

    # Merge forecast data with QC data
    qc_comparison = agg_by_year_df.merge(
        qc_data,
        on='SKU',
        how='outer',
        suffixes=('', '_QC')
    )

    # Calculate deltas
    qc_comparison['Delta_2024'] = qc_comparison['2024'] - qc_comparison['sales_2024'].fillna(0)
    qc_comparison['Delta_2025'] = qc_comparison['2025'] - qc_comparison['sales_2025'].fillna(0)
    qc_comparison['Delta_2026_YTD'] = qc_comparison['2026_YTD'] - qc_comparison['sales_2026'].fillna(0)

    # Reorder columns for clarity
    qc_comparison = qc_comparison[[
        'SKU', 'Product_Name', 'Is_Active',
        '2024', 'sales_2024', 'Delta_2024',
        '2025', 'sales_2025', 'Delta_2025',
        '2026_YTD', 'sales_2026', 'Delta_2026_YTD',
        '2026_Forecast'
    ]]

    # Fill NaN with 0 for cleaner display
    qc_comparison = qc_comparison.fillna(0).astype({
        '2024': int, 'sales_2024': int, 'Delta_2024': int,
        '2025': int, 'sales_2025': int, 'Delta_2025': int,
        '2026_YTD': int, 'sales_2026': int, 'Delta_2026_YTD': int,
        '2026_Forecast': int
    })

    print(f"✅ QC comparison built: {len(qc_comparison)} SKUs")
    print(f"\nSummary:")
    print(f"  Total Delta 2024: {qc_comparison['Delta_2024'].sum():,}")
    print(f"  Total Delta 2025: {qc_comparison['Delta_2025'].sum():,}")
    print(f"  Total Delta 2026 YTD: {qc_comparison['Delta_2026_YTD'].sum():,}")

    # Write QC sheet
    print("\nWriting QC Check sheet...")
    ws_qc = sh.add_worksheet(title='QC Check', rows=1000, cols=15)

    header = [
        'SKU', 'Product Name', 'Is Active',
        'Forecast 2024', 'QC 2024', 'Delta 2024',
        'Forecast 2025', 'QC 2025', 'Delta 2025',
        'Forecast 2026 YTD', 'QC 2026', 'Delta 2026 YTD',
        'Forecast 2026 Total'
    ]

    data_rows = [header] + qc_comparison.values.tolist()
    ws_qc.update('A1', data_rows)

    # Format header
    ws_qc.format('A1:M1', {
        'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
        'backgroundColor': {'red': 0.259, 'green': 0.522, 'blue': 0.957}
    })

    # Highlight delta columns in yellow
    ws_qc.format('F:F', {'backgroundColor': {'red': 1, 'green': 1, 'blue': 0.8}})  # Delta 2024
    ws_qc.format('I:I', {'backgroundColor': {'red': 1, 'green': 1, 'blue': 0.8}})  # Delta 2025
    ws_qc.format('L:L', {'backgroundColor': {'red': 1, 'green': 1, 'blue': 0.8}})  # Delta 2026 YTD

    print(f"  ✅ QC Check sheet created with {len(data_rows)} rows")
else:
    print("\n⚠️  Skipping QC comparison (no QC data uploaded)")



Building QC comparison...
✅ QC comparison built: 277 SKUs

Summary:
  Total Delta 2024: 31,681
  Total Delta 2025: 22,361
  Total Delta 2026 YTD: -12,084

Writing QC Check sheet...
  ✅ QC Check sheet created with 278 rows


In [55]:
# Final output
print("\n" + "="*80)
print("DEMAND PLANNING TOOL v5 CREATED SUCCESSFULLY!")
print("="*80)
print(f"\nGoogle Sheet Name: {sheet_name}")
print(f"URL: https://docs.google.com/spreadsheets/d/{sh.id}")
print(f"\nSheets created:")
for worksheet in sh.worksheets():
    print(f"  - {worksheet.title}")
print(f"\nTotal SKUs analyzed: {len(sku_details)}")
print(f"Channels: {', '.join(all_channels)}")
print(f"Forecast period: {forecast_months[0]} — {forecast_months[-1]}")
print(f"\nWhat's new in v3:")
print(f"  ✅ Seasonality by PRODUCT TYPE (24-month pooled indices, not per-item)")
print(f"  ✅ Seasonality table in Product Type Breakdown tab (month × product type)")
print(f"  ✅ Forecast sheets show 2024 + 2025 actuals alongside 2026 forecast")
print(f"  ✅ Actual vs Forecast columns color-coded (slate = actual, green = forecast)")
print(f"  ✅ Channel total row added to each forecast sheet")
print(f"  ✅ Aggregated By Month — 3 channel tabs (Wholesale / DTC / TOTAL), SKU × month rows")
print(f"  ✅ Building Blocks tab: Raw 2025 | Seasonal Index | Deseasonalized | Base | Trend | Forecast")
print(f"  ✅ Inactivity rule: SKUs with 0 sales in last 6 months excluded from all forecasts")
print(f"  ✅ Per-channel growth rate overrides (CHANNEL_GROWTH_RATES dict)")



DEMAND PLANNING TOOL v5 CREATED SUCCESSFULLY!

Google Sheet Name: Demand_Planning_20260309_1758
URL: https://docs.google.com/spreadsheets/d/1GmyyJ8fgq6OXerlYYfyEBMu-isQ2uY6Xiyq3POMafLA

Sheets created:
  - Summary Dashboard
  - Export
  - Product Type Breakdown
  - Direct-to-Consumer - Forecast
  - Wholesale - Forecast
  - TOTAL - Forecast
  - Agg by Month — Direct-to-Consumer
  - Agg by Month — Wholesale
  - Agg by Month — TOTAL
  - Building Blocks
  - Pareto Analysis
  - QC Check

Total SKUs analyzed: 277
Channels: Direct-to-Consumer, Wholesale, TOTAL
Forecast period: 2026-03 — 2026-12

What's new in v3:
  ✅ Seasonality by PRODUCT TYPE (24-month pooled indices, not per-item)
  ✅ Seasonality table in Product Type Breakdown tab (month × product type)
  ✅ Forecast sheets show 2024 + 2025 actuals alongside 2026 forecast
  ✅ Actual vs Forecast columns color-coded (slate = actual, green = forecast)
  ✅ Channel total row added to each forecast sheet
  ✅ Aggregated By Month — 3 channel tabs